# PST and GS-ST timing on GPU and CPU

Benchmark of matched PST and GS-ST implementations in separate GPU and CPU processes. The CPU profile uses one eighth of the GPU trajectory counts; timing comparisons are made within each device profile.

Both primary profiles are enabled by default. Missing timing results trigger measurements; `FORCE_RERUN = False` reuses existing timing JSON files. Set it to `True` after changing a workload or to collect new measurements.

Run from the `Timing/` directory using the `PST (Python 3.11)` kernel. See [Running the notebooks](../docs/RUNNING.md) for execution settings and result reuse.

**Authors:** Jose M. G. Vilar$^{1,2}$ and Leonor Saiz$^{3}$  
$^1$ Biofisika Institute (CSIC, UPV/EHU), University of the Basque Country, P.O. Box 644, 48080 Bilbao, Spain  
$^2$ IKERBASQUE, Basque Foundation for Science, 48011 Bilbao, Spain  
$^3$ Department of Biomedical Engineering, University of California, 451 East Health Sciences Drive, Davis, CA 95616, USA

**Preprint:** J. M. G. Vilar and L. Saiz, *Propensity Straight-Through Gradients for Discrete Stochastic Systems*, arXiv:2608.25631 [q-bio.QM] (2026).  
**DOI:** [https://doi.org/10.48550/arXiv.2608.25631](https://doi.org/10.48550/arXiv.2608.25631)


## Methods and device profiles

Each device profile measures the same set of implementations in a fresh Python process. The GPU profile uses the full workload; the CPU profile uses a smaller workload.

Methods measured on both devices:

- **PST-cat** — original TensorFlow categorical PST implementation.
- **PST-SSA** — direct inverse-CDF SSA hard sampler from raw propensities with the analytic affine PST backward rule; no reaction-selection logits or softmax.
- **PST-SSA-XLA** — the same PST-SSA estimator, with XLA compilation applied to the detached hard inverse-CDF sampler. The outer simulator, loss, reverse-mode differentiation, optimizer, and PST VJP remain non-XLA.
- **PST-GM** — explicit Gumbel-Max hard sampler with the PST backward rule.
- **GSST-GM** — the same explicit Gumbel-Max hard sampler with the Gumbel-Softmax backward rule.
- **SSA-fwd** — exact forward simulation plus loss only; a lower-bound reference for training-step cost.

The primary matched-sampler estimator comparison is **PST-GM / GSST-GM**. The practical implementation comparisons are **PST-SSA / GSST-GM** and **PST-SSA-XLA / GSST-GM**. **PST-SSA-XLA / PST-SSA** isolates the effect of XLA on the direct hard sampler.

The GPU uses 262,144 ion-channel trajectories and 8,192 oscillator starting states with 25 trajectories per state (204,800 trajectories). The CPU uses 32,768 ion-channel trajectories and 1,024 oscillator starting states with 25 trajectories per state (25,600 trajectories), one eighth of the GPU trajectory counts. Both devices use 20 ion-channel event slots and 150 oscillator events. Compare methods within each device profile, since absolute GPU and CPU times reflect different workload sizes.

Timing repeats are multiples of the six rows (18 on GPU, 12 on CPU), so the cyclic timing order is exactly balanced.

Run from the `Timing/` directory. The experimental XLSX input is included; reference trajectories and timing results are generated on first use. Both primary profiles are enabled by default. Missing timing JSON files trigger fresh measurements; on later runs, `FORCE_RERUN = False` reuses the generated JSON files when present. Set `FORCE_RERUN = True` to collect fresh measurements again. GPU selection uses `PST_BENCH_GPU_ID`, then `CUDA_VISIBLE_DEVICES`, and defaults to device `0`; set `GPU_ID = "auto"` to leave device visibility unchanged. For CPU-only execution, set `RUN_GPU_PRIMARY = False`. The optional single-thread CPU diagnostic is disabled by default.

When needed, the oscillator reference is generated automatically and saved for later reuse: `reference_trajectory.npz` contains the 2,400,000-event GPU reference, and `reference_trajectory_cpu_quick.npz` contains the 600,000-event CPU reference. Reference generation occurs outside the timed intervals. The CPU filename does not indicate a separate execution mode. Timing CSV and JSON files are written beside the notebook.


In [ ]:
from pathlib import Path
import os
import sys
import json
import subprocess
import pandas as pd
import numpy as np

# =============================================================================
# USER SETTINGS
# =============================================================================

RUN_GPU_PRIMARY = True
RUN_CPU_PRIMARY = True

# Optional robustness diagnostic; not needed for the main CPU table.
RUN_CPU_SINGLE_THREAD_DIAGNOSTIC = False

# Respect caller GPU selection; otherwise default to device 0.
# Set GPU_ID = "auto" to leave CUDA_VISIBLE_DEVICES unchanged.
GPU_ID = os.environ.get("PST_BENCH_GPU_ID", os.environ.get("CUDA_VISIBLE_DEVICES", "0"))

ION_DATA_FILE = Path("HEK293_Cell01_40mV_100s_raw_DET_IDEl.xlsx")

RUN_ION = True
RUN_CLOCK = True
FORCE_RERUN = False

# Primary GPU: full workload, three complete balanced timing cycles.
GPU_REPEATS = 18
GPU_WARMUP = 3

# Primary CPU: one eighth of the GPU trajectory counts, two balanced cycles.
CPU_REPEATS = 12
CPU_WARMUP = 2

if RUN_ION and not ION_DATA_FILE.exists():
    raise FileNotFoundError(
        f"Ion-channel data file not found: {ION_DATA_FILE.name}\n"
        "Place the XLSX next to this notebook or edit ION_DATA_FILE above."
    )

print("Driver Python:", sys.version.split()[0])
print("Working directory: notebook directory")


In [ ]:
# Self-contained worker. Each benchmark condition launches this source in a
# fresh Python process so GPU visibility and CPU thread pools are fixed before
# TensorFlow initializes.
WORKER_SOURCE = '"""\nPST / GS-ST coherent GPU + CPU sampler-decomposition timing worker\n==================================================================\n\nThis worker is launched by the parent notebook in a fresh Python process for\neach device/profile so TensorFlow device visibility and CPU thread pools are\nfixed before runtime initialization.\n\nTimed methods:\n* PST-cat: original TensorFlow categorical hard draw + PST backward.\n* PST-SSA: direct inverse-CDF SSA hard draw from raw propensities + analytic\n  affine PST VJP; no reaction-selection logits or softmax.\n* PST-SSA-XLA: exactly the same PST-SSA estimator, but ONLY the detached hard\n  inverse-CDF sampler is XLA-compiled. The outer simulator, loss, GradientTape,\n  optimizer update, and PST custom VJP remain non-XLA.\n* PST-GM: explicit Gumbel-Max hard draw + PST backward.\n* GSST-GM: the same explicit Gumbel-Max hard draw + Gumbel-Softmax backward.\n* SSA-fwd: exact forward simulation + loss only; no backward or optimizer.\n\nThe primary estimator-cost control is PST-GM/GSST-GM because the hard Gumbel-Max\nforward construction is matched. PST-SSA and PST-SSA-XLA measure practical\ndirect-propensity implementations. XLA is isolated as an implementation\noptimization rather than changing the estimator.\n\nAll timed training rows include forward simulation, readout/loss, reverse-mode\nbackpropagation, gradient clipping/checks, optimizer update, and the clock\nparameter projection. Tracing/warm-up, parameter/optimizer resets, clock batch\nselection, and reference generation are outside timed intervals.\n"""\n\nfrom __future__ import annotations\n\nimport os\n\n# -----------------------------------------------------------------------------\n# DEVICE / PROCESS SETTINGS -- supplied by the parent notebook before TF import.\n# -----------------------------------------------------------------------------\nBENCHMARK_PROFILE = os.environ.get("PST_BENCH_LABEL", "gpu_primary")\nBENCHMARK_DEVICE = os.environ.get("PST_BENCH_DEVICE", "gpu").strip().lower()\nGPU_ID = os.environ.get("PST_BENCH_GPU_ID", "0").strip()\nCPU_THREADS = os.environ.get("PST_BENCH_CPU_THREADS", "auto").strip().lower()\n\nif BENCHMARK_DEVICE not in {"gpu", "cpu"}:\n    raise ValueError("PST_BENCH_DEVICE must be \'gpu\' or \'cpu\'")\n\nif BENCHMARK_DEVICE == "cpu":\n    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"\n    if CPU_THREADS != "auto":\n        n_threads = int(CPU_THREADS)\n        if n_threads < 1:\n            raise ValueError("CPU thread count must be positive")\n        os.environ["OMP_NUM_THREADS"] = str(n_threads)\n        os.environ["TF_NUM_INTRAOP_THREADS"] = str(n_threads)\n        os.environ["TF_NUM_INTEROP_THREADS"] = "1"\nelse:\n    # Empty/\'auto\' leaves CUDA visibility unchanged. Otherwise expose one GPU.\n    if GPU_ID and GPU_ID.lower() != "auto":\n        os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID\n\nos.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")\n\nimport sys\nimport time\nimport json\nimport math\nimport random\nimport platform\nimport statistics\nfrom pathlib import Path\nfrom dataclasses import dataclass, asdict\n\nimport numpy as np\nimport pandas as pd\nimport tensorflow as tf\n\n# Explicit CPU thread control must occur before any TensorFlow computation.\nif BENCHMARK_DEVICE == "cpu" and CPU_THREADS != "auto":\n    _n_threads = int(CPU_THREADS)\n    tf.config.threading.set_intra_op_parallelism_threads(_n_threads)\n    tf.config.threading.set_inter_op_parallelism_threads(1)\n\n_physical_gpus = tf.config.list_physical_devices("GPU")\nif BENCHMARK_DEVICE == "gpu" and not _physical_gpus:\n    raise RuntimeError(\n        "GPU benchmark requested, but TensorFlow sees no physical GPU. "\n        "Check GPU_ID/CUDA_VISIBLE_DEVICES before interpreting this run."\n    )\nif BENCHMARK_DEVICE == "cpu" and _physical_gpus:\n    raise RuntimeError(\n        "CPU benchmark requested, but a GPU is still visible to TensorFlow."\n    )\n\n\n# =============================================================================\n# USER SETTINGS -- supplied by parent notebook via environment variables\n# =============================================================================\n\nRUN_CLOCK = os.environ.get("PST_RUN_CLOCK", "1") == "1"\nRUN_ION = os.environ.get("PST_RUN_ION", "1") == "1"\n\nION_DATA_FILE = Path(os.environ.get(\n    "PST_ION_DATA_FILE",\n    "HEK293_Cell01_40mV_100s_raw_DET_IDEl.xlsx",\n))\n\nION_NUM_STEPS = int(os.environ.get("PST_ION_NUM_STEPS", "20"))\nION_EPOCHS = 400\nCLOCK_SEGMENT_STEPS = int(os.environ.get("PST_CLOCK_SEGMENT_STEPS", "150"))\nCLOCK_M_TRAIN = int(os.environ.get("PST_CLOCK_M_TRAIN", "25"))\n\nION_M_SIMS = int(os.environ["PST_ION_M_SIMS"])\nCLOCK_NUM_START_POINTS = int(os.environ["PST_CLOCK_NUM_START_POINTS"])\nCLOCK_REFERENCE_STEPS = int(os.environ["PST_CLOCK_REFERENCE_STEPS"])\nCLOCK_BURN_IN = int(os.environ["PST_CLOCK_BURN_IN"])\nCLOCK_BATCH_SIZE = int(os.environ["PST_CLOCK_BATCH_SIZE"])\nWARMUP = int(os.environ["PST_BENCH_WARMUP"])\nREPEATS = int(os.environ["PST_BENCH_REPEATS"])\nSSA_DIAGNOSTIC_DRAWS = int(os.environ["PST_SSA_DIAGNOSTIC_DRAWS"])\nCLOCK_REFERENCE_FILE = Path(os.environ["PST_CLOCK_REFERENCE_FILE"])\n\n# Reference generation is outside timing. A common ion-readout XLA setting may\n# be enabled for the GPU profile, but it is identical across all timed methods.\n# PST-SSA-XLA remains the only METHOD-SPECIFIC XLA difference.\nREFERENCE_JIT_COMPILE = os.environ.get("PST_REFERENCE_JIT", "0") == "1"\nION_READOUT_JIT = os.environ.get("PST_ION_READOUT_JIT", "0") == "1"\n\nBASE_SEED = 43\nSHOW_PROGRESS = os.environ.get("PST_SHOW_PROGRESS", "1") == "1"\nION_TIMING_EPOCH = 200\n\nPST_CAT_NAME = "PST-cat"\nPST_GM_NAME = "PST-GM"\nGSST_GM_NAME = "GSST-GM"\nPST_DIRECT_NAME = "PST-SSA"\nPST_XLA_NAME = "PST-SSA-XLA"\nSSA_METHOD_NAME = "SSA-fwd"\n\nINCLUDE_PST_DIRECT = True\nINCLUDE_PST_XLA = True\nINCLUDE_SSA_BASELINE = True\n\nOUTPUT_CSV = Path(f"pst_gsst_timing_{BENCHMARK_PROFILE}.csv")\nOUTPUT_JSON = Path(f"pst_gsst_timing_{BENCHMARK_PROFILE}.json")\n\n\n# =============================================================================\n# ENVIRONMENT / HELPERS\n# =============================================================================\n\ndef configure_devices():\n    for gpu in tf.config.list_physical_devices("GPU"):\n        try:\n            tf.config.experimental.set_memory_growth(gpu, True)\n        except (RuntimeError, ValueError):\n            pass\n\nconfigure_devices()\n\n\ndef environment_metadata():\n    build = {}\n    try:\n        build = tf.sysconfig.get_build_info()\n    except Exception:\n        pass\n    return {\n        "benchmark_profile": BENCHMARK_PROFILE,\n        "benchmark_device": BENCHMARK_DEVICE,\n        "gpu_id_requested": GPU_ID,\n        "cpu_threads_requested": CPU_THREADS,\n        "tf_intra_op_threads": tf.config.threading.get_intra_op_parallelism_threads(),\n        "tf_inter_op_threads": tf.config.threading.get_inter_op_parallelism_threads(),\n        "python": sys.version.split()[0],\n        "platform": platform.platform(),\n        "processor": platform.processor(),\n        "os_cpu_count": os.cpu_count(),\n        "tensorflow": tf.__version__,\n        "numpy": np.__version__,\n        "physical_cpus": [x.name for x in tf.config.list_physical_devices("CPU")],\n        "physical_gpus": [x.name for x in tf.config.list_physical_devices("GPU")],\n        "logical_gpus": [x.name for x in tf.config.list_logical_devices("GPU")],\n        "cuda_version": build.get("cuda_version"),\n        "cudnn_version": build.get("cudnn_version"),\n        "is_cuda_build": build.get("is_cuda_build"),\n    }\n\n\ndef print_environment():\n    print("\\nEnvironment")\n    print("-----------")\n    for k, v in environment_metadata().items():\n        print(f"{k:>16}: {v}")\n\n\ndef sync_tensor(x):\n    """Materialize returned values and therefore synchronize device work.\n\n    TensorFlow tensors/variables are copied to host with .numpy(). Some\n    Python-side training steps (notably the ion-channel step) already return a\n    NumPy array after performing their own .numpy() synchronization; those are\n    simply passed through.\n    """\n    if isinstance(x, (tuple, list)):\n        return [sync_tensor(v) for v in x]\n    if isinstance(x, dict):\n        return {k: sync_tensor(v) for k, v in x.items()}\n    if isinstance(x, np.ndarray):\n        return x\n    if isinstance(x, np.generic):\n        return np.asarray(x)\n    if tf.is_tensor(x) or isinstance(x, tf.Variable):\n        return np.asarray(x.numpy())\n    return np.asarray(x)\n\n\ndef snapshot_variables(variables):\n    return [np.array(v.numpy(), copy=True) for v in variables]\n\n\ndef restore_variables(variables, values):\n    for v, value in zip(variables, values):\n        v.assign(value)\n\n\ndef optimizer_snapshot(optimizer):\n    return snapshot_variables(list(optimizer.variables))\n\n\ndef optimizer_restore(optimizer, values):\n    restore_variables(list(optimizer.variables), values)\n\n\n@dataclass\nclass TimingResult:\n    model: str\n    method: str\n    trajectories: int\n    ssa_steps: int\n    trajectory_events: int\n    repeats: int\n    median_ms: float\n    mean_ms: float\n    std_ms: float\n    min_ms: float\n    max_ms: float\n    ns_per_trajectory_event: float\n    million_trajectory_events_per_s: float\n    samples_ms: list[float]\n\n\n@dataclass\nclass PairedStats:\n    model: str\n    numerator: str\n    denominator: str\n    repeats: int\n    median_ratio: float\n    ratio_ci_low: float\n    ratio_ci_high: float\n    mean_diff_ms: float\n    sd_diff_ms: float\n    t_statistic: float\n    degrees_of_freedom: int\n    numerator_slower_count: int\n\n\ndef summarize(model, method, trajectories, ssa_steps, samples):\n    samples = [float(x) for x in samples]\n    med = statistics.median(samples)\n    events = trajectories * ssa_steps\n    return TimingResult(\n        model=model,\n        method=method,\n        trajectories=trajectories,\n        ssa_steps=ssa_steps,\n        trajectory_events=events,\n        repeats=len(samples),\n        median_ms=med,\n        mean_ms=statistics.mean(samples),\n        std_ms=statistics.stdev(samples) if len(samples) > 1 else 0.0,\n        min_ms=min(samples),\n        max_ms=max(samples),\n        ns_per_trajectory_event=(med * 1e6) / events,\n        million_trajectory_events_per_s=events / (med * 1e3),\n        samples_ms=samples,\n    )\n\n\ndef _format_seconds(seconds):\n    seconds = max(0.0, float(seconds))\n    if seconds < 60:\n        return f"{seconds:.1f}s"\n    minutes, sec = divmod(seconds, 60)\n    if minutes < 60:\n        return f"{int(minutes)}m {sec:04.1f}s"\n    hours, minutes = divmod(minutes, 60)\n    return f"{int(hours)}h {int(minutes):02d}m {sec:04.1f}s"\n\n\ndef time_paired(methods, resetters, callers, warmup, repeats, base_seed):\n    """\n    methods: ordered method labels to time within each repeat\n    resetters[name](): restore identical parameters and optimizer state\n    callers[name](seed_tensor): execute one full stochastic gradient update\n\n    Progress output is outside the timed interval and reports completion of\n    each warm-up and measured call.\n    """\n    samples = {m: [] for m in methods}\n    total_calls = warmup * len(methods) + repeats * len(methods)\n    completed_calls = 0\n    overall_start = time.perf_counter()\n\n    print(f"  warm-up={warmup}, timed repeats={repeats}", flush=True)\n    print(f"  total expensive calls={total_calls}", flush=True)\n\n    # -------------------------\n    # Warm-up / graph tracing\n    # -------------------------\n    for w in range(warmup):\n        for m in methods:\n            resetters[m]()\n            seed = tf.constant([base_seed, 10_000 + w], tf.int32)\n\n            if SHOW_PROGRESS:\n                print(\n                    f"  [warm-up {w + 1:02d}/{warmup:02d}] "\n                    f"{m:<7} ... ",\n                    end="",\n                    flush=True,\n                )\n\n            call_start = time.perf_counter()\n            sync_tensor(callers[m](seed))\n            call_elapsed = time.perf_counter() - call_start\n\n            completed_calls += 1\n            if SHOW_PROGRESS:\n                elapsed = time.perf_counter() - overall_start\n                avg_call = elapsed / completed_calls\n                remaining = (total_calls - completed_calls) * avg_call\n                print(\n                    f"done in {_format_seconds(call_elapsed)} "\n                    f"| overall {completed_calls}/{total_calls} "\n                    f"| rough ETA {_format_seconds(remaining)}",\n                    flush=True,\n                )\n\n    if SHOW_PROGRESS:\n        print("  Warm-up complete. Starting timed repetitions.", flush=True)\n\n    # -------------------------\n    # Timed paired repetitions\n    # -------------------------\n    timed_start = time.perf_counter()\n\n    for r in range(repeats):\n        # Balanced cyclic order. Over every len(methods) repeats, each\n        # method occupies every ordinal position exactly once. Alternate the\n        # direction in successive blocks to reduce carry-over/thermal bias.\n        n_methods = len(methods)\n        shift = r % n_methods\n        order = list(methods[shift:]) + list(methods[:shift])\n        if (r // n_methods) % 2 == 1:\n            order = list(reversed(order))\n        seed = tf.constant([base_seed, 100_000 + r], tf.int32)\n        pair_values = {}\n\n        for m in order:\n            resetters[m]()\n\n            if SHOW_PROGRESS:\n                print(\n                    f"  [repeat {r + 1:02d}/{repeats:02d}] "\n                    f"{m:<7} ... ",\n                    end="",\n                    flush=True,\n                )\n\n            # Only this interval contributes to the reported timing.\n            t0 = time.perf_counter_ns()\n            out = callers[m](seed)\n            sync_tensor(out)\n            t1 = time.perf_counter_ns()\n\n            ms = (t1 - t0) / 1e6\n            samples[m].append(ms)\n            pair_values[m] = ms\n            completed_calls += 1\n\n            if SHOW_PROGRESS:\n                print(f"{ms:,.1f} ms", flush=True)\n\n        if SHOW_PROGRESS:\n            timed_elapsed = time.perf_counter() - timed_start\n            repeats_done = r + 1\n            mean_repeat_time = timed_elapsed / repeats_done\n            eta = (repeats - repeats_done) * mean_repeat_time\n\n            pair_text = " | ".join(\n                f"{m}={pair_values[m]:,.1f} ms"\n                for m in methods\n                if m in pair_values\n            )\n            median_text = ", ".join(\n                f"{m}={statistics.median(samples[m]):,.1f} ms"\n                for m in methods\n                if samples[m]\n            )\n            print(\n                f"    completed set {r + 1:02d}/{repeats:02d}: {pair_text}\\n"\n                f"    running medians: {median_text}"\n                f" | ETA {_format_seconds(eta)}",\n                flush=True,\n            )\n\n    if SHOW_PROGRESS:\n        print(\n            f"  Timing complete in "\n            f"{_format_seconds(time.perf_counter() - timed_start)}.",\n            flush=True,\n        )\n\n    return samples\n\n\n# =============================================================================\n# CLOCK / VILAR OSCILLATOR — COPIED FROM THE TRAINING PROTOCOL\n# =============================================================================\n\nCLOCK_K_TRUE = np.array([\n    1.0, 50.0, 1.0, 100.0,\n    50.0, 500.0, 0.01, 50.0,\n    50.0, 5.0, 10.0, 0.5,\n    1.0, 0.2, 2.0, 1.0,\n], dtype=np.float32)\n\nCLOCK_S = tf.constant([\n    [-1., +1.,  0.,  0.,  0.,  0., -1.,  0.,  0.],\n    [+1., -1.,  0.,  0.,  0.,  0., +1.,  0.,  0.],\n    [ 0.,  0., -1., +1.,  0.,  0., -1.,  0.,  0.],\n    [ 0.,  0., +1., -1.,  0.,  0., +1.,  0.,  0.],\n    [ 0.,  0.,  0.,  0., +1.,  0.,  0.,  0.,  0.],\n    [ 0.,  0.,  0.,  0., +1.,  0.,  0.,  0.,  0.],\n    [ 0.,  0.,  0.,  0.,  0., +1.,  0.,  0.,  0.],\n    [ 0.,  0.,  0.,  0.,  0., +1.,  0.,  0.,  0.],\n    [ 0.,  0.,  0.,  0.,  0.,  0., +1.,  0.,  0.],\n    [ 0.,  0.,  0.,  0.,  0.,  0.,  0., +1.,  0.],\n    [ 0.,  0.,  0.,  0., -1.,  0.,  0.,  0.,  0.],\n    [ 0.,  0.,  0.,  0.,  0., -1.,  0.,  0.,  0.],\n    [ 0.,  0.,  0.,  0.,  0.,  0., -1.,  0.,  0.],\n    [ 0.,  0.,  0.,  0.,  0.,  0.,  0., -1.,  0.],\n    [ 0.,  0.,  0.,  0.,  0.,  0., -1., -1., +1.],\n    [ 0.,  0.,  0.,  0.,  0.,  0.,  0., +1., -1.],\n], tf.float32)\n\n\n@tf.function(jit_compile=False, reduce_retracing=True)\ndef clock_propensity(states, k):\n    X = tf.maximum(states, 1e-10)\n    DA, DAp, DR, DRp = X[:, 0], X[:, 1], X[:, 2], X[:, 3]\n    MA, MR, A, R, C = X[:, 4], X[:, 5], X[:, 6], X[:, 7], X[:, 8]\n    return tf.stack([\n        k[0] * DA * A,   k[1] * DAp,\n        k[2] * DR * A,   k[3] * DRp,\n        k[4] * DA,       k[5] * DAp,\n        k[6] * DR,       k[7] * DRp,\n        k[8] * MA,       k[9] * MR,\n        k[10] * MA,      k[11] * MR,\n        k[12] * A,       k[13] * R,\n        k[14] * A * R,   k[15] * C,\n    ], axis=1)\n\n\ndef _clock_gumbel_max(logits, seed):\n    """Exact Gumbel-Max hard categorical draw shared by PST-GM/GSST-GM."""\n    u = tf.random.stateless_uniform(\n        tf.shape(logits),\n        seed=seed,\n        minval=1e-20,\n        maxval=1.0,\n        dtype=logits.dtype,\n    )\n    g = -tf.math.log(-tf.math.log(u))\n    hard_idx = tf.argmax(logits + g, axis=-1)\n    y_hard = tf.one_hot(\n        hard_idx,\n        depth=tf.shape(logits)[-1],\n        dtype=logits.dtype,\n    )\n    return y_hard, g\n\n\ndef clock_pst_categorical_estimator(logits, seed):\n    """Original oscillator PST implementation.\n\n    Forward: exact hard draw via tf.random.stateless_categorical.\n    Backward: derivative of pi = softmax(logits).\n\n    The flatten/reshape is intentionally retained because it is part of the\n    TensorFlow categorical API path being measured.\n    """\n    logits_shape = tf.shape(logits)\n    num_reactions = logits_shape[-1]\n    flat_logits = tf.reshape(logits, [-1, num_reactions])\n    hard_idx_flat = tf.random.stateless_categorical(\n        flat_logits,\n        num_samples=1,\n        seed=seed,\n    )\n    hard_idx = tf.reshape(hard_idx_flat, logits_shape[:-1])\n    y_hard = tf.one_hot(\n        hard_idx,\n        depth=num_reactions,\n        dtype=logits.dtype,\n    )\n    probability = tf.nn.softmax(logits, axis=-1)\n    return probability + tf.stop_gradient(y_hard - probability)\n\n\ndef clock_pst_gumbel_estimator(logits, seed):\n    """Exact Gumbel-Max hard forward; PST backward.\n\n    Gumbel noise is used only for the hard categorical realization. The\n    backward derivative is d pi, with no temperature or Gumbel-dependent\n    relaxation.\n    """\n    y_hard, _ = _clock_gumbel_max(logits, seed)\n    probability = tf.nn.softmax(logits, axis=-1)\n    return probability + tf.stop_gradient(y_hard - probability)\n\n\ndef clock_gsst_gumbel_estimator(logits, tau, seed):\n    """Exact Gumbel-Max hard forward; Gumbel-Softmax ST backward."""\n    y_hard, g = _clock_gumbel_max(logits, seed)\n    y_soft = tf.nn.softmax((logits + g) / tau, axis=-1)\n    return y_soft + tf.stop_gradient(y_hard - y_soft)\n\n\ndef make_clock_simulator(method):\n    """One oscillator loop, shared by both estimators.\n\n    Only the reaction-selection call differs.  Propensities, absorbing-state\n    masking, waiting-time sampling, the state update and the tensor-array\n    bookkeeping are identical between the two branches, so a timing\n    difference between the traced graphs is attributable to the estimator and\n    not to how the surrounding loop happens to be written.\n\n    The absorbing-state guard is retained for BOTH methods.  It is needed\n    whenever a row can reach a0 = 0: a softmax over an all -inf logit row is\n    NaN and the sampled index is arbitrary, so the fallback keeps one entry\n    positive and the trailing mask zeroes the resulting weights.  The same guard is applied to both estimators.\n    """\n    assert method in {PST_CAT_NAME, PST_GM_NAME, PST_DIRECT_NAME, PST_XLA_NAME, GSST_GM_NAME}\n\n    @tf.function(jit_compile=False, reduce_retracing=True)\n    def simulate(batch_initial_states, num_steps, k, temperature, S_matrix, seed):\n        num_simulations = tf.shape(batch_initial_states)[0]\n        time_t = tf.zeros([num_simulations, 1], tf.float32)\n        states_ta = tf.TensorArray(tf.float32, size=num_steps + 1)\n        times_ta = tf.TensorArray(tf.float32, size=num_steps + 1)\n        states_ta = states_ta.write(0, batch_initial_states)\n        times_ta = times_ta.write(0, time_t)\n\n        t = tf.constant(0)\n        states = batch_initial_states\n\n        def cond(t, states, time_t, states_ta, times_ta, seed):\n            return t < num_steps\n\n        def body(t, states, time_t, states_ta, times_ta, seed):\n            seeds = tf.random.experimental.stateless_split(seed, 3)\n            next_seed, seed_time, seed_reaction = seeds[0], seeds[1], seeds[2]\n\n            prop = clock_propensity(states, k)\n            a0 = tf.reduce_sum(prop, axis=1, keepdims=True)\n            active = a0 > 0.0\n            safe_a0 = tf.where(active, a0, tf.ones_like(a0))\n\n            u_time = tf.random.stateless_uniform(\n                [num_simulations, 1],\n                seed=seed_time,\n                minval=1e-20,\n                maxval=1.0,\n                dtype=prop.dtype,\n            )\n            dt = tf.where(\n                active,\n                -tf.math.log(u_time) / safe_a0,\n                tf.zeros_like(u_time),\n            )\n\n            fallback = tf.one_hot(\n                tf.zeros([num_simulations], dtype=tf.int32),\n                depth=tf.shape(prop)[-1],\n                dtype=prop.dtype,\n            )\n            sampling_prop = tf.where(active, prop, fallback)\n\n            if method in {PST_DIRECT_NAME, PST_XLA_NAME}:\n                # Both PST-SSA variants use the same exact direct-method\n                # forward and the same analytic PST affine VJP.\n                #\n                # PST-SSA-XLA differs ONLY by routing the detached\n                # CDF/compare/reduce/gather hard sampler through an\n                # XLA-compiled subfunction.\n                delta = clock_pst_delta_ssa(\n                    sampling_prop,\n                    safe_a0,\n                    S_matrix,\n                    seed_reaction,\n                    use_xla_sampler=(method == PST_XLA_NAME),\n                )\n                delta = tf.where(active, delta, tf.zeros_like(delta))\n            else:\n                # Logits are needed only by the categorical/Gumbel variants.\n                positive = sampling_prop > 0.0\n                finite_log = tf.math.log(\n                    tf.where(\n                        positive,\n                        sampling_prop,\n                        tf.ones_like(sampling_prop),\n                    )\n                )\n                logits = tf.where(\n                    positive,\n                    finite_log,\n                    tf.constant(float("-inf"), dtype=prop.dtype),\n                )\n\n                if method == PST_CAT_NAME:\n                    weights = clock_pst_categorical_estimator(\n                        logits, seed_reaction\n                    )\n                elif method == PST_GM_NAME:\n                    weights = clock_pst_gumbel_estimator(\n                        logits, seed_reaction\n                    )\n                else:\n                    weights = clock_gsst_gumbel_estimator(\n                        logits, temperature, seed_reaction\n                    )\n\n                weights = tf.where(active, weights, tf.zeros_like(weights))\n                delta = tf.matmul(weights, S_matrix)\n\n            new_states = tf.maximum(states + delta, 0.0)\n            new_time = time_t + dt\n\n            states_ta = states_ta.write(t + 1, new_states)\n            times_ta = times_ta.write(t + 1, new_time)\n            return t + 1, new_states, new_time, states_ta, times_ta, next_seed\n\n        _, _, _, states_ta, times_ta, _ = tf.while_loop(\n            cond, body,\n            [t, states, time_t, states_ta, times_ta, seed],\n            shape_invariants=[\n                t.shape,\n                tf.TensorShape([None, 9]),\n                tf.TensorShape([None, 1]),\n                tf.TensorShape([]),\n                tf.TensorShape([]),\n                tf.TensorShape([2]),\n            ],\n        )\n        states = tf.transpose(states_ta.stack(), [1, 0, 2])\n        times = tf.transpose(times_ta.stack(), [1, 0, 2])\n        return states, times\n\n    return simulate\n\n\n\ndef verify_clock_matched_gumbel_forward():\n    """Check PST-GM and GSST-GM have the same hard-forward trajectory."""\n    print("\\nMatched Gumbel-Max forward check")\n    print("--------------------------------")\n\n    sim_pst = make_clock_simulator(PST_GM_NAME)\n    sim_gs = make_clock_simulator(GSST_GM_NAME)\n\n    x0 = tf.repeat(\n        tf.constant([[1., 0., 1., 0., 0., 0., 0., 0., 0.]], tf.float32),\n        repeats=32,\n        axis=0,\n    )\n    k = tf.constant(CLOCK_K_TRUE, tf.float32)\n    seed = tf.constant([123, 456], tf.int32)\n    steps = tf.constant(20, tf.int32)\n    tau = tf.constant(3e-6, tf.float32)\n\n    s_pst, t_pst = sim_pst(x0, steps, k, tau, CLOCK_S, seed)\n    s_gs, t_gs = sim_gs(x0, steps, k, tau, CLOCK_S, seed)\n\n    state_err = float(tf.reduce_max(tf.abs(s_pst - s_gs)).numpy())\n    time_err = float(tf.reduce_max(tf.abs(t_pst - t_gs)).numpy())\n\n    print(f"  max |state PST-GM - GSST-GM| = {state_err:.3e}")\n    print(f"  max |time  PST-GM - GSST-GM| = {time_err:.3e}")\n\n    if state_err > 1e-5 or time_err > 1e-5:\n        raise AssertionError(\n            "Matched Gumbel-Max methods did not produce the same forward "\n            f"trajectory: state_err={state_err}, time_err={time_err}"\n        )\n    print("  matched hard-forward construction confirmed.")\n\n\ndef clock_reference_core(initial_states, num_steps, k, seed):\n    """\n    Fast exact-forward reference trajectory core.\n\n    This deliberately follows the ORIGINAL oscillator notebook design:\n      * reference/target generation: XLA JIT compiled (no gradients needed)\n      * differentiable PST/GS-ST training: jit_compile=False\n\n    Although the PST straight-through tensor is constructed, its numerical\n    forward value is exactly the hard categorical one-hot, so the generated\n    reference path is an exact SSA path.\n    """\n    num_simulations = tf.shape(initial_states)[0]\n    time_t = tf.zeros([num_simulations, 1], tf.float32)\n\n    states_ta = tf.TensorArray(tf.float32, size=num_steps + 1)\n    times_ta = tf.TensorArray(tf.float32, size=num_steps + 1)\n    states_ta = states_ta.write(0, initial_states)\n    times_ta = times_ta.write(0, time_t)\n\n    t = tf.constant(0)\n    states = initial_states\n\n    def cond(t, states, time_t, states_ta, times_ta, seed):\n        return t < num_steps\n\n    def body(t, states, time_t, states_ta, times_ta, seed):\n        seeds = tf.random.experimental.stateless_split(seed, 3)\n        next_seed = seeds[0]\n        seed_time = seeds[1]\n        seed_reaction = seeds[2]\n\n        prop = clock_propensity(states, k)\n        a0 = tf.reduce_sum(prop, axis=1, keepdims=True)\n        active = a0 > 0.0\n        safe_a0 = tf.where(active, a0, tf.ones_like(a0))\n\n        u_time = tf.random.stateless_uniform(\n            [num_simulations, 1],\n            seed=seed_time,\n            minval=1e-20,\n            maxval=1.0,\n            dtype=prop.dtype,\n        )\n        dt = tf.where(\n            active,\n            -tf.math.log(u_time) / safe_a0,\n            tf.zeros_like(u_time),\n        )\n\n        # Same exact-forward reaction-selection construction as the original\n        # PST oscillator notebook.\n        fallback = tf.one_hot(\n            tf.zeros([num_simulations], dtype=tf.int32),\n            depth=tf.shape(prop)[-1],\n            dtype=prop.dtype,\n        )\n        sampling_prop = tf.where(active, prop, fallback)\n        positive = sampling_prop > 0.0\n        finite_log = tf.math.log(\n            tf.where(positive, sampling_prop, tf.ones_like(sampling_prop))\n        )\n        logits = tf.where(\n            positive,\n            finite_log,\n            tf.constant(float("-inf"), dtype=prop.dtype),\n        )\n\n        weights = clock_pst_categorical_estimator(logits, seed_reaction)\n        weights = tf.where(active, weights, tf.zeros_like(weights))\n\n        delta = tf.matmul(weights, CLOCK_S)\n        new_states = tf.maximum(states + delta, 0.0)\n        new_time = time_t + dt\n\n        states_ta = states_ta.write(t + 1, new_states)\n        times_ta = times_ta.write(t + 1, new_time)\n        return (\n            t + 1,\n            new_states,\n            new_time,\n            states_ta,\n            times_ta,\n            next_seed,\n        )\n\n    _, _, _, states_ta, times_ta, _ = tf.while_loop(\n        cond,\n        body,\n        [t, states, time_t, states_ta, times_ta, seed],\n        shape_invariants=[\n            t.shape,\n            tf.TensorShape([None, 9]),\n            tf.TensorShape([None, 1]),\n            tf.TensorShape([]),\n            tf.TensorShape([]),\n            tf.TensorShape([2]),\n        ],\n    )\n\n    return (\n        tf.transpose(states_ta.stack(), [1, 0, 2]),\n        tf.transpose(times_ta.stack(), [1, 0, 2]),\n    )\n\n\n# Reference generation is outside timed intervals.  The GPU-full profile uses\n# the original XLA path; the CPU profile disables this unrelated JIT for\n# portability and lower compilation overhead.\nclock_reference_fast = tf.function(\n    clock_reference_core,\n    jit_compile=REFERENCE_JIT_COMPILE,\n    reduce_retracing=True,\n)\n\n\ndef load_or_generate_clock_reference(path=CLOCK_REFERENCE_FILE):\n    if path.exists():\n        print(f"Loading clock reference trajectory: {Path(path).name}")\n        with np.load(path) as z:\n            states = np.asarray(z["states"], dtype=np.float32)\n            times = np.asarray(z["times"], dtype=np.float32)\n        if states.ndim == 3:\n            states = states[0]\n        if times.ndim == 3:\n            times = times[0]\n        return tf.constant(states), tf.constant(times)\n\n    print(\n        f"{Path(path).name} not found. Generating {CLOCK_REFERENCE_STEPS:,} exact "\n        "reference events once (not timed).",\n        flush=True,\n    )\n    print(\n        f"  Reference-generator XLA JIT: {REFERENCE_JIT_COMPILE}.",\n        flush=True,\n    )\n\n    init = tf.constant([[1., 0., 1., 0., 0., 0., 0., 0., 0.]], tf.float32)\n    k_ref = tf.constant(CLOCK_K_TRUE, tf.float32)\n\n    # Compile/warm the XLA graph with the same input signatures first.\n    print("  Compiling/warming fast reference simulator ... ", end="", flush=True)\n    warm_start = time.perf_counter()\n    warm_states, warm_times = clock_reference_fast(\n        init,\n        tf.constant(10, tf.int32),\n        k_ref,\n        tf.constant([BASE_SEED, 10], tf.int32),\n    )\n    sync_tensor((warm_states[:, -1, :], warm_times[:, -1, :]))\n    print(\n        f"done in {_format_seconds(time.perf_counter() - warm_start)}.",\n        flush=True,\n    )\n\n    print(\n        f"  Generating full {CLOCK_REFERENCE_STEPS:,}-event reference ... ",\n        end="",\n        flush=True,\n    )\n    ref_start = time.perf_counter()\n    states, times = clock_reference_fast(\n        init,\n        tf.constant(CLOCK_REFERENCE_STEPS, tf.int32),\n        k_ref,\n        tf.constant([BASE_SEED, 11], tf.int32),\n    )\n    sync_tensor((states[:, -1, :], times[:, -1, :]))\n    print(\n        f"done in {_format_seconds(time.perf_counter() - ref_start)}.",\n        flush=True,\n    )\n    states_np = states.numpy()\n    times_np = times.numpy()\n    np.savez(path, states=states_np, times=times_np)\n    return tf.constant(states_np[0]), tf.constant(times_np[0])\n\n\ndef prepare_clock_batch():\n    """\n    Reproduce the protocol\'s stratified target-map construction, but materialize\n    only the start-state batch configured for the selected benchmark profile.\n    """\n    reference_states, reference_times = load_or_generate_clock_reference()\n\n    needed = CLOCK_REFERENCE_STEPS + 1\n    if int(reference_states.shape[0]) < needed:\n        raise ValueError(\n            f"Reference trajectory has {reference_states.shape[0]} stored states; "\n            f"need at least {needed}."\n        )\n\n    tf.random.set_seed(BASE_SEED)\n\n    burn_in = CLOCK_BURN_IN\n    max_start_excl = CLOCK_REFERENCE_STEPS - CLOCK_SEGMENT_STEPS + 1\n    N = tf.constant(CLOCK_NUM_START_POINTS, tf.int32)\n\n    edges = tf.linspace(\n        tf.cast(burn_in, tf.float32),\n        tf.cast(max_start_excl, tf.float32),\n        N + 1,\n    )\n    left, right = edges[:-1], edges[1:]\n    u = tf.random.uniform([N], 0.0, 1.0)\n    map_start_indices = tf.cast(tf.floor(left + u * (right - left)), tf.int32)\n    map_start_indices = tf.clip_by_value(\n        map_start_indices, burn_in, max_start_excl - 1\n    )\n\n    all_map_indices = tf.range(CLOCK_NUM_START_POINTS, dtype=tf.int32)\n    chosen = tf.random.shuffle(all_map_indices)[:CLOCK_BATCH_SIZE]\n    start_indices = tf.gather(map_start_indices, chosen)\n\n    offsets = tf.range(CLOCK_SEGMENT_STEPS + 1, dtype=tf.int32)[None, :]\n    trajectory_indices = start_indices[:, None] + offsets\n\n    target_states = tf.gather(reference_states, trajectory_indices)\n    target_times_raw = tf.gather(reference_times, trajectory_indices)\n    target_times = target_times_raw - target_times_raw[:, :1, :]\n\n    initial_states = tf.gather(reference_states, start_indices)\n    initial_states = tf.repeat(\n        initial_states[:, None, :], repeats=CLOCK_M_TRAIN, axis=1\n    )\n\n    initial_states = tf.ensure_shape(\n        initial_states, [CLOCK_BATCH_SIZE, CLOCK_M_TRAIN, 9]\n    )\n    target_states = tf.ensure_shape(\n        target_states, [CLOCK_BATCH_SIZE, CLOCK_SEGMENT_STEPS + 1, 9]\n    )\n    target_times = tf.ensure_shape(\n        target_times, [CLOCK_BATCH_SIZE, CLOCK_SEGMENT_STEPS + 1, 1]\n    )\n\n    print("Clock timing batch prepared:")\n    print(f"  starting states: {CLOCK_BATCH_SIZE:,}")\n    print(f"  trajectories/start: {CLOCK_M_TRAIN}")\n    print(f"  total trajectories: {CLOCK_BATCH_SIZE * CLOCK_M_TRAIN:,}")\n    print(f"  events/trajectory: {CLOCK_SEGMENT_STEPS}")\n    return initial_states, target_states, target_times\n\n\ndef clock_compute_loss(sim_states, target_states, sim_times, target_times):\n    epsilon = 1e-9\n\n    def rate(states, times, feature_idx):\n        mean_final_state = tf.reduce_mean(\n            states[:, -100:, feature_idx], axis=1, keepdims=True\n        )\n        mean_final_time = tf.reduce_mean(\n            times[:, -100:, 0], axis=1, keepdims=True\n        )\n        delta_state = mean_final_state - states[:, 0:1, feature_idx]\n        delta_time = mean_final_time - times[:, 0:1, 0]\n        return delta_state / (delta_time + epsilon)\n\n    loss_A = tf.reduce_mean(tf.square(\n        rate(sim_states, sim_times, 6) - rate(target_states, target_times, 6)\n    ))\n    loss_R = tf.reduce_mean(tf.square(\n        rate(sim_states, sim_times, 7) - rate(target_states, target_times, 7)\n    ))\n    loss_C = tf.reduce_mean(tf.square(\n        rate(sim_states, sim_times, 8) - rate(target_states, target_times, 8)\n    ))\n    return loss_A + loss_R + loss_C\n\n\ndef make_clock_trainer(method):\n    """\n    Actual clock train_step workload: simulation + rate loss + reverse-mode\n    gradient + clipping + finite check + RMSprop + true-centered projection.\n    """\n    assert method in {PST_CAT_NAME, PST_GM_NAME, PST_DIRECT_NAME, PST_XLA_NAME, GSST_GM_NAME}\n\n    true_params = tf.constant([50., 5., 1., 0.2, 2.], tf.float32)\n    # Same parameters for both methods during timing.\n    initial_params = true_params\n\n    variables = [\n        tf.Variable(tf.math.log(initial_params[0]), name=f"{method}_log_beta_A"),\n        tf.Variable(tf.math.log(initial_params[1]), name=f"{method}_log_beta_R"),\n        tf.Variable(tf.math.log(initial_params[2]), name=f"{method}_log_delta_A"),\n        tf.Variable(tf.math.log(initial_params[3]), name=f"{method}_log_delta_R"),\n        tf.Variable(tf.math.log(initial_params[4]), name=f"{method}_log_gamma_C"),\n    ]\n    optimizer = tf.keras.optimizers.RMSprop(learning_rate=0.1)\n    optimizer.build(variables)\n\n    simulate = make_clock_simulator(method)\n\n    initial_var_state = snapshot_variables(variables)\n    initial_opt_state = optimizer_snapshot(optimizer)\n\n    true_log = tf.math.log(true_params)\n    true_log_list = tf.unstack(true_log)\n    log10 = tf.math.log(tf.constant(10.0, tf.float32))\n    k_const = tf.constant(CLOCK_K_TRUE, tf.float32)\n    tau = tf.constant(3e-6, tf.float32)\n\n    @tf.function(\n        jit_compile=False,\n        reduce_retracing=True,\n        input_signature=[\n            tf.TensorSpec([CLOCK_BATCH_SIZE, CLOCK_M_TRAIN, 9], tf.float32),\n            tf.TensorSpec([CLOCK_BATCH_SIZE, CLOCK_SEGMENT_STEPS + 1, 9], tf.float32),\n            tf.TensorSpec([CLOCK_BATCH_SIZE, CLOCK_SEGMENT_STEPS + 1, 1], tf.float32),\n            tf.TensorSpec([2], tf.int32),\n        ],\n    )\n    def train_step(batch_initial_states, batch_target_states, batch_target_times, seed2):\n        with tf.GradientTape() as tape:\n            p = [tf.exp(v) for v in variables]\n            k_learn = tf.concat([\n                k_const[:8],\n                [p[0]], [p[1]],\n                k_const[10:12],\n                [p[2]], [p[3]], [p[4]], [p[2]],\n            ], axis=0)\n\n            super_batch_initial_states = tf.repeat(\n                batch_initial_states[:, 0, :],\n                repeats=CLOCK_M_TRAIN,\n                axis=0,\n            )\n\n            super_states, super_times = simulate(\n                super_batch_initial_states,\n                tf.constant(CLOCK_SEGMENT_STEPS, tf.int32),\n                k_learn,\n                tau,\n                CLOCK_S,\n                seed2,\n            )\n\n            sim_states = tf.reshape(\n                super_states,\n                [CLOCK_BATCH_SIZE, CLOCK_M_TRAIN, CLOCK_SEGMENT_STEPS + 1, 9],\n            )\n            sim_times = tf.reshape(\n                super_times,\n                [CLOCK_BATCH_SIZE, CLOCK_M_TRAIN, CLOCK_SEGMENT_STEPS + 1, 1],\n            )\n            mean_states = tf.reduce_mean(sim_states, axis=1)\n            mean_times = tf.reduce_mean(sim_times, axis=1)\n\n            loss = clock_compute_loss(\n                mean_states, batch_target_states, mean_times, batch_target_times\n            )\n\n        grads = tape.gradient(loss, variables)\n        grads, _ = tf.clip_by_global_norm(grads, 1.0)\n\n        # Original pre-update metrics.\n        p_pre = [tf.exp(v) for v in variables]\n        rpd = tf.norm(tf.stack(variables) - true_log)\n        mape = tf.reduce_mean(\n            tf.abs((tf.stack(p_pre) - true_params) / (true_params + 1e-9)) * 100\n        )\n\n        grads_finite = tf.reduce_all(\n            [tf.reduce_all(tf.math.is_finite(g)) for g in grads]\n        )\n        ok = tf.math.is_finite(loss) & grads_finite\n\n        def apply_update():\n            optimizer.apply_gradients(zip(grads, variables))\n            for v, truth in zip(variables, true_log_list):\n                v.assign(tf.clip_by_value(v, truth - log10, truth + log10))\n            return tf.constant(True)\n\n        did_update = tf.cond(ok, apply_update, lambda: tf.constant(False))\n\n        # Force the returned value to depend on the optimizer update/projection.\n        with tf.control_dependencies([did_update]):\n            post_checksum = tf.add_n([tf.reduce_sum(v.read_value()) for v in variables])\n\n        return tf.stack([\n            loss,\n            rpd,\n            mape,\n            post_checksum,\n            tf.cast(did_update, tf.float32),\n        ])\n\n    def reset():\n        restore_variables(variables, initial_var_state)\n        optimizer_restore(optimizer, initial_opt_state)\n\n    return train_step, reset\n\n\n\ndef benchmark_clock():\n    print("\\n" + "=" * 78)\n    print("CLOCK: SAMPLER / BACKWARD-RULE DECOMPOSITION")\n    print("=" * 78)\n\n    print("Preparing the real oscillator timing batch ...", flush=True)\n    batch_initial, batch_target_states, batch_target_times = prepare_clock_batch()\n\n    print("Building oscillator training graphs ...", flush=True)\n    cat_step, cat_reset = make_clock_trainer(PST_CAT_NAME)\n    gm_step, gm_reset = make_clock_trainer(PST_GM_NAME)\n    gs_step, gs_reset = make_clock_trainer(GSST_GM_NAME)\n\n    check_clock_graph_parity(gm_step, gs_step)\n\n    def clock_caller(step_fn):\n        return lambda seed: step_fn(\n            batch_initial, batch_target_states, batch_target_times, seed\n        )\n\n    methods = [PST_CAT_NAME]\n    resetters = {PST_CAT_NAME: cat_reset}\n    callers = {PST_CAT_NAME: clock_caller(cat_step)}\n\n    if INCLUDE_PST_DIRECT:\n        print(\n            "Building PST-SSA oscillator graph "\n            "(unfused direct SSA forward + analytic affine PST VJP) ...",\n            flush=True,\n        )\n        ssa_pst_step, ssa_pst_reset = make_clock_trainer(PST_DIRECT_NAME)\n        methods.append(PST_DIRECT_NAME)\n        resetters[PST_DIRECT_NAME] = ssa_pst_reset\n        callers[PST_DIRECT_NAME] = clock_caller(ssa_pst_step)\n\n    if INCLUDE_PST_XLA:\n        print(\n            "Building PST-SSA-XLA oscillator graph "\n            "(XLA hard sampler only; outer training remains non-XLA) ...",\n            flush=True,\n        )\n        ssa_xla_step, ssa_xla_reset = make_clock_trainer(PST_XLA_NAME)\n        methods.append(PST_XLA_NAME)\n        resetters[PST_XLA_NAME] = ssa_xla_reset\n        callers[PST_XLA_NAME] = clock_caller(ssa_xla_step)\n\n    methods.extend([PST_GM_NAME, GSST_GM_NAME])\n    resetters.update({\n        PST_GM_NAME: gm_reset,\n        GSST_GM_NAME: gs_reset,\n    })\n    callers.update({\n        PST_GM_NAME: clock_caller(gm_step),\n        GSST_GM_NAME: clock_caller(gs_step),\n    })\n\n    if INCLUDE_SSA_BASELINE:\n        print("Building exact SSA forward-only oscillator graph ...", flush=True)\n        ssa_step, ssa_reset = make_clock_ssa_forward()\n        methods.append(SSA_METHOD_NAME)\n        resetters[SSA_METHOD_NAME] = ssa_reset\n        callers[SSA_METHOD_NAME] = clock_caller(ssa_step)\n\n    samples = time_paired(\n        methods, resetters, callers, WARMUP, REPEATS, BASE_SEED\n    )\n\n    ntraj = CLOCK_BATCH_SIZE * CLOCK_M_TRAIN\n    return [\n        summarize("clock", m, ntraj, CLOCK_SEGMENT_STEPS, samples[m])\n        for m in methods\n    ]\n\n\n# =============================================================================\n# ION CHANNEL — COPIED FROM THE TRAINING PROTOCOL\n# =============================================================================\n\nION_S = tf.constant([\n    [-1.0, +1.0, 0.0],\n    [+1.0, -1.0, 0.0],\n    [0.0, -1.0, +1.0],\n], tf.float32)\n\nION_N_CHANNELS = 2\n\n\n@tf.function(jit_compile=False, reduce_retracing=True)\ndef ion_propensity(states, k_open, k_close, k_inact):\n    states = tf.maximum(states, 0.0)\n    n_closed = states[:, 0]\n    n_open = states[:, 1]\n    return tf.stack([\n        k_open * n_closed,\n        k_close * n_open,\n        k_inact * n_open,\n    ], axis=1)\n\n\ndef _ion_gumbel_max(logits):\n    """Exact stateful Gumbel-Max hard categorical draw."""\n    u = tf.random.uniform(\n        tf.shape(logits),\n        minval=1e-20,\n        maxval=1.0,\n        dtype=logits.dtype,\n    )\n    g = -tf.math.log(-tf.math.log(u))\n    hard_idx = tf.argmax(logits + g, axis=-1)\n    y_hard = tf.one_hot(\n        hard_idx,\n        depth=tf.shape(logits)[-1],\n        dtype=logits.dtype,\n    )\n    return y_hard, g\n\n\ndef ion_pst_categorical_estimator(logits):\n    """Original ion-channel PST: categorical hard forward, d pi backward."""\n    probabilities = tf.nn.softmax(logits, axis=-1)\n    logits_shape = tf.shape(logits)\n    flat_logits = tf.reshape(logits, [-1, logits_shape[-1]])\n    hard_idx = tf.random.categorical(flat_logits, 1)\n    hard_idx = tf.reshape(hard_idx, logits_shape[:-1])\n    y_hard = tf.one_hot(\n        hard_idx,\n        depth=logits_shape[-1],\n        dtype=logits.dtype,\n    )\n    return probabilities + tf.stop_gradient(y_hard - probabilities)\n\n\ndef ion_pst_gumbel_estimator(logits):\n    """Exact Gumbel-Max hard forward; PST backward."""\n    y_hard, _ = _ion_gumbel_max(logits)\n    probabilities = tf.nn.softmax(logits, axis=-1)\n    return probabilities + tf.stop_gradient(y_hard - probabilities)\n\n\ndef ion_gsst_gumbel_estimator(logits, tau):\n    """Exact Gumbel-Max hard forward; Gumbel-Softmax ST backward."""\n    y_hard, g = _ion_gumbel_max(logits)\n    y_soft = tf.nn.softmax((logits + g) / tau, axis=-1)\n    return y_soft + tf.stop_gradient(y_hard - y_soft)\n\ndef make_ion_simulator(method):\n    assert method in {PST_CAT_NAME, PST_GM_NAME, PST_DIRECT_NAME, PST_XLA_NAME, GSST_GM_NAME}\n\n    @tf.function(jit_compile=False, reduce_retracing=True)\n    def simulate(batch_initial_states, k_open, k_close, k_inact, temperature):\n        num_sims = tf.shape(batch_initial_states)[0]\n        time_t = tf.zeros([num_sims, 1], tf.float32)\n        states_ta = tf.TensorArray(tf.float32, size=ION_NUM_STEPS + 1)\n        times_ta = tf.TensorArray(tf.float32, size=ION_NUM_STEPS + 1)\n        states_ta = states_ta.write(0, batch_initial_states)\n        times_ta = times_ta.write(0, time_t)\n\n        states = batch_initial_states\n        step = tf.constant(0)\n\n        def cond(step, states, time_t, states_ta, times_ta):\n            return step < ION_NUM_STEPS\n\n        def body(step, states, time_t, states_ta, times_ta):\n            prop = ion_propensity(states, k_open, k_close, k_inact)\n            a0 = tf.reduce_sum(prop, axis=1, keepdims=True)\n\n            absorbing = a0 < 1e-10\n            safe_a0 = tf.where(absorbing, tf.ones_like(a0), a0)\n\n            u_time = tf.random.uniform(\n                [num_sims, 1],\n                minval=1e-20,\n                maxval=1.0,\n                dtype=tf.float32,\n            )\n            dt = -tf.math.log(u_time) / safe_a0\n            dt = tf.where(absorbing, tf.ones_like(dt) * 100000.0, dt)\n\n            # Preserve exact zero-propensity support. Absorbing rows get\n            # one dummy positive channel solely to keep sampling well defined;\n            # the update is masked to zero below.\n            fallback = tf.one_hot(\n                tf.zeros([num_sims], dtype=tf.int32),\n                depth=tf.shape(prop)[-1],\n                dtype=prop.dtype,\n            )\n            sampling_prop = tf.where(absorbing, fallback, prop)\n\n            if method in {PST_DIRECT_NAME, PST_XLA_NAME}:\n                # Same PST estimator in both cases.  PST-SSA-XLA only\n                # XLA-compiles the detached hard direct-method sampler.\n                delta = ion_pst_delta_ssa(\n                    sampling_prop,\n                    safe_a0,\n                    ION_S,\n                    use_xla_sampler=(method == PST_XLA_NAME),\n                )\n            else:\n                positive = sampling_prop > 0.0\n                finite_log = tf.math.log(\n                    tf.where(\n                        positive,\n                        sampling_prop,\n                        tf.ones_like(sampling_prop),\n                    )\n                )\n                logits = tf.where(\n                    positive,\n                    finite_log,\n                    tf.constant(float("-inf"), dtype=prop.dtype),\n                )\n\n                if method == PST_CAT_NAME:\n                    weights = ion_pst_categorical_estimator(logits)\n                elif method == PST_GM_NAME:\n                    weights = ion_pst_gumbel_estimator(logits)\n                else:\n                    weights = ion_gsst_gumbel_estimator(\n                        logits, temperature\n                    )\n\n                delta = tf.matmul(weights, ION_S)\n\n            delta = tf.where(absorbing, tf.zeros_like(delta), delta)\n\n            states = tf.maximum(states + delta, 0.0)\n            time_t = time_t + dt\n\n            states_ta = states_ta.write(step + 1, states)\n            times_ta = times_ta.write(step + 1, time_t)\n            return step + 1, states, time_t, states_ta, times_ta\n\n        _, _, _, states_ta, times_ta = tf.while_loop(\n            cond,\n            body,\n            [step, states, time_t, states_ta, times_ta],\n            shape_invariants=[\n                step.shape,\n                tf.TensorShape([None, 3]),\n                tf.TensorShape([None, 1]),\n                tf.TensorShape([]),\n                tf.TensorShape([]),\n            ],\n        )\n        states = tf.transpose(states_ta.stack(), [1, 0, 2])\n        times = tf.transpose(times_ta.stack(), [1, 0, 2])\n        return states, times\n\n    return simulate\n\n\n@tf.function(jit_compile=ION_READOUT_JIT, reduce_retracing=True)\ndef ion_bin_integrated_readout(sim_t, sim_v, tgt_t):\n    sim_t = tf.squeeze(sim_t, axis=-1)\n    n_sims = tf.shape(sim_t)[0]\n    n_steps = tf.shape(sim_t)[1]\n    dtype = sim_t.dtype\n\n    ramp = tf.cast(tf.range(n_steps)[None, :], dtype) * tf.cast(1e-8, dtype)\n    sim_t = sim_t + ramp\n\n    dt = tf.maximum(sim_t[:, 1:] - sim_t[:, :-1], tf.cast(0.0, dtype))\n    step_areas = sim_v[:, :-1] * dt\n    cumulative_area = tf.concat(\n        [tf.zeros([n_sims, 1], dtype), tf.cumsum(step_areas, axis=1)],\n        axis=1,\n    )\n\n    first_t = sim_t[:, :1]\n    last_t = sim_t[:, -1:]\n    last_area = cumulative_area[:, -1:]\n    last_value = sim_v[:, -1:]\n\n    def cumulative_at(query_t):\n        n_query = tf.shape(query_t)[0]\n        q = tf.broadcast_to(query_t[None, :], [n_sims, n_query])\n        q = tf.cast(q, dtype)\n\n        idx = tf.searchsorted(sim_t, q, side="right") - 1\n        idx = tf.clip_by_value(idx, 0, n_steps - 2)\n\n        area0 = tf.gather(cumulative_area, idx, batch_dims=1)\n        time0 = tf.gather(sim_t, idx, batch_dims=1)\n        value0 = tf.gather(sim_v, idx, batch_dims=1)\n        inside = area0 + value0 * (q - time0)\n\n        before = tf.zeros_like(q)\n        after = last_area + last_value * (q - last_t)\n        return tf.where(q < first_t, before, tf.where(q >= last_t, after, inside))\n\n    area_start = cumulative_at(tgt_t[:-1])\n    area_end = cumulative_at(tgt_t[1:])\n    widths = tf.cast(tgt_t[1:] - tgt_t[:-1], dtype)\n    return (area_end - area_start) / widths[None, :]\n\n\ndef make_ion_loss(method):\n    simulate = make_ion_simulator(method)\n\n    @tf.function(jit_compile=False, reduce_retracing=True)\n    def loss_fn(initial_state, t_target, n_open_target,\n                k_open, k_close, k_inact, temperature):\n        super_batch = tf.repeat(initial_state, repeats=ION_M_SIMS, axis=0)\n        sim_states, sim_times = simulate(\n            super_batch, k_open, k_close, k_inact, temperature\n        )\n\n        n_open_sim = sim_states[:, :, 1]\n        model_bin_avg = ion_bin_integrated_readout(\n            sim_times, n_open_sim, t_target\n        )\n\n        model_bin_avg = tf.reshape(\n            model_bin_avg,\n            [1, ION_M_SIMS, tf.shape(t_target)[0] - 1],\n        )\n        model_mean = tf.reduce_mean(model_bin_avg, axis=[0, 1])\n        experimental_bin_avg = 0.5 * (\n            n_open_target[:-1] + n_open_target[1:]\n        )\n        return tf.reduce_mean(tf.square(model_mean - experimental_bin_avg))\n\n    return loss_fn\n\n\ndef load_ion_data(path=ION_DATA_FILE):\n    if not path.exists():\n        raise FileNotFoundError(\n            f"Ion-channel data file not found: {Path(path).name}\\n"\n            "Edit ION_DATA_FILE at the top of the notebook to point to the "\n            "original HEK293 XLSX file. A synthetic replacement is intentionally "\n            "not used in this faithful benchmark."\n        )\n    df = pd.read_excel(path, header=None, skiprows=5)\n    time_ms = df.iloc[:, 0].values.astype(np.float32)\n    sweeps = df.iloc[:, 1:].values.astype(np.float32)\n    mean_open = np.nanmean(sweeps, axis=1).astype(np.float32)\n    time_shifted = (time_ms - time_ms[0]).astype(np.float32)\n    print("Ion-channel data loaded:")\n    print(f"  sweeps={sweeps.shape[1]}, timepoints={len(time_ms)}")\n    return tf.constant(time_shifted), tf.constant(mean_open)\n\n\ndef make_ion_trainer(method, t_target, n_open_target):\n    """\n    Mirrors the original ion training loop: GradientTape in Python, loss graph,\n    gradient materialization for NaN checking, then RMSprop update.\n    """\n    assert method in {PST_CAT_NAME, PST_GM_NAME, PST_DIRECT_NAME, PST_XLA_NAME, GSST_GM_NAME}\n    loss_fn = make_ion_loss(method)\n\n    log_k_open = tf.Variable(tf.math.log(0.25), name=f"{method}_ion_log_open")\n    log_k_close = tf.Variable(tf.math.log(0.25), name=f"{method}_ion_log_close")\n    log_k_inact = tf.Variable(tf.math.log(0.25), name=f"{method}_ion_log_inact")\n    variables = [log_k_open, log_k_close, log_k_inact]\n\n    optimizer = tf.keras.optimizers.RMSprop(learning_rate=0.1)\n    optimizer.build(variables)\n\n    initial_var_state = snapshot_variables(variables)\n    initial_opt_state = optimizer_snapshot(optimizer)\n\n    initial_state = tf.constant([[2.0, 0.0, 0.0]], tf.float32)\n\n    schedule_value = 0.1 * (0.001 / 0.1) ** (\n        ION_TIMING_EPOCH / max(1, ION_EPOCHS - 1)\n    )\n    temperature = tf.constant(schedule_value, tf.float32)\n    learning_rate = 0.05 * (schedule_value / 0.1)\n\n    def reset():\n        restore_variables(variables, initial_var_state)\n        optimizer_restore(optimizer, initial_opt_state)\n\n    def step(_seed_unused):\n        # The ion-channel training notebooks use stateful TensorFlow RNG, so the seed\n        # argument is intentionally unused here.\n        optimizer.learning_rate.assign(learning_rate)\n\n        with tf.GradientTape() as tape:\n            k_open = tf.exp(log_k_open)\n            k_close = tf.exp(log_k_close)\n            k_inact = tf.exp(log_k_inact)\n            loss = loss_fn(\n                initial_state,\n                t_target,\n                n_open_target,\n                k_open,\n                k_close,\n                k_inact,\n                temperature,\n            )\n\n        gradients = tape.gradient(loss, variables)\n\n        # Match the original Python-side NaN check; .numpy() synchronizes the\n        # gradient before the optimizer update, exactly as the training loop did.\n        invalid_gradient = any(\n            g is None or bool(tf.reduce_any(tf.math.is_nan(g)).numpy())\n            for g in gradients\n        )\n        if not invalid_gradient:\n            optimizer.apply_gradients(zip(gradients, variables))\n\n        # Force completion of the update and match original scalar history reads.\n        values = np.array([\n            float(loss.numpy()),\n            float(tf.exp(log_k_open).numpy()),\n            float(tf.exp(log_k_close).numpy()),\n            float(tf.exp(log_k_inact).numpy()),\n            float(optimizer.iterations.numpy()),\n        ], dtype=np.float64)\n        return values\n\n    return step, reset\n\n\n\ndef benchmark_ion():\n    print("\\n" + "=" * 78)\n    print("ION CHANNEL: SAMPLER / BACKWARD-RULE DECOMPOSITION")\n    print("=" * 78)\n\n    t_target, n_open_target = load_ion_data()\n\n    print("Building ion-channel training graphs ...", flush=True)\n    cat_step, cat_reset = make_ion_trainer(\n        PST_CAT_NAME, t_target, n_open_target\n    )\n    gm_step, gm_reset = make_ion_trainer(\n        PST_GM_NAME, t_target, n_open_target\n    )\n    gs_step, gs_reset = make_ion_trainer(\n        GSST_GM_NAME, t_target, n_open_target\n    )\n\n    methods = [PST_CAT_NAME]\n    resetters = {PST_CAT_NAME: cat_reset}\n    callers = {PST_CAT_NAME: cat_step}\n\n    if INCLUDE_PST_DIRECT:\n        print(\n            "Building PST-SSA ion-channel graph "\n            "(unfused direct SSA forward + analytic affine PST VJP) ...",\n            flush=True,\n        )\n        ssa_pst_step, ssa_pst_reset = make_ion_trainer(\n            PST_DIRECT_NAME, t_target, n_open_target\n        )\n        methods.append(PST_DIRECT_NAME)\n        resetters[PST_DIRECT_NAME] = ssa_pst_reset\n        callers[PST_DIRECT_NAME] = ssa_pst_step\n\n    if INCLUDE_PST_XLA:\n        print(\n            "Building PST-SSA-XLA ion-channel graph "\n            "(XLA hard sampler only; outer training remains non-XLA) ...",\n            flush=True,\n        )\n        ssa_xla_step, ssa_xla_reset = make_ion_trainer(\n            PST_XLA_NAME, t_target, n_open_target\n        )\n        methods.append(PST_XLA_NAME)\n        resetters[PST_XLA_NAME] = ssa_xla_reset\n        callers[PST_XLA_NAME] = ssa_xla_step\n\n    methods.extend([PST_GM_NAME, GSST_GM_NAME])\n    resetters.update({\n        PST_GM_NAME: gm_reset,\n        GSST_GM_NAME: gs_reset,\n    })\n    callers.update({\n        PST_GM_NAME: gm_step,\n        GSST_GM_NAME: gs_step,\n    })\n\n    if INCLUDE_SSA_BASELINE:\n        print("Building exact SSA forward-only ion graph ...", flush=True)\n        ssa_step, ssa_reset = make_ion_ssa_forward(\n            t_target, n_open_target\n        )\n        methods.append(SSA_METHOD_NAME)\n        resetters[SSA_METHOD_NAME] = ssa_reset\n        callers[SSA_METHOD_NAME] = ssa_step\n\n    samples = time_paired(\n        methods, resetters, callers, WARMUP, REPEATS, BASE_SEED\n    )\n\n    return [\n        summarize("ion", m, ION_M_SIMS, ION_NUM_STEPS, samples[m])\n        for m in methods\n    ]\n\n\n# =============================================================================\n# STANDARD GILLESPIE SSA (DIRECT METHOD)\n# =============================================================================\n#\n# Reaction selection by inverse-transform sampling on the cumulative\n# propensity (Gillespie 1977): draw u ~ U[0,1), take the smallest j with\n# cumsum(a)[j] > u * a0.  No categorical kernel, no Gumbel noise.\n#\n# This serves two distinct purposes and they must not be conflated:\n#\n#   (1) A non-differentiable exact-SSA forward (clock_simulate_ssa /\n#       ion_simulate_ssa) timed WITHOUT a tape or optimizer, reported as\n#       "SSA-fwd".  It is a lower bound on simulation cost, not a training\n#       step, and is not comparable to the PST/GS-ST rows.\n#\n#   (2) A direct-method forward inside the PST estimator ("PST-dm").  The\n#       backward branch is bit-identical to PST, so this remains a full\n#       training step and stays comparable to GS-ST.  GS-ST has no such\n#       variant: its backward relaxation needs the Gumbel variates, so it\n#       cannot avoid drawing them.  That asymmetry is a property of the\n#       estimators, not of the implementation.\n\n# Graph-level assertions on the sampler.  MUST stay False for timed runs: the\n# assert op sits inside the inner while_loop body and is not free.  Read at\n# trace time, so changing it requires rebuilding the graphs.\nSSA_DEBUG_CHECKS = False\n\n# False -> state update is tf.gather(S, index), what a real SSA does.\n# True  -> state update is one_hot(index) @ S, matching the PST/GS-ST loop body\n#          op-for-op so the timing difference isolates the sampler alone.\nSSA_ONEHOT_STATE_UPDATE = False\n\n# Method labels and inclusion flags are defined with the main settings.\n\n\ndef _apply_reaction(reaction_index, S_matrix, num_reactions, dtype):\n    """State delta for a sampled reaction index; see SSA_ONEHOT_STATE_UPDATE."""\n    if SSA_ONEHOT_STATE_UPDATE:\n        weights = tf.one_hot(reaction_index, depth=num_reactions, dtype=dtype)\n        return tf.matmul(weights, S_matrix)\n    return tf.gather(S_matrix, reaction_index)\n\n\ndef direct_method_index(propensities, u):\n    """Smallest j with cumsum(propensities)[j] > u * a0, per row.\n\n    propensities : [B, R] float, non-negative, row sum STRICTLY POSITIVE.\n    u            : [B, 1] float in [0, 1).\n    returns      : [B] int32 in [0, R-1].\n\n    Precondition: every row must have a positive sum.  Absorbing rows must be\n    masked by the CALLER before this call -- see the `fallback` construction in\n    the simulators.  A dead row reaching here returns R, which is out of range\n    for tf.gather, and SSA_DEBUG_CHECKS detects this invalid input.\n\n    a0 is taken as the last cumsum element rather than a separate reduce_sum so\n    that `threshold <= a0 = cdf[R-1]` is exact.  Recomputing the total with\n    reduce_sum can let threshold exceed cdf[R-1] by an ulp through rounding\n    and push the index to R.\n\n    On the returned index being in range, which matters because tf.gather with\n    an out-of-range index raises on CPU but silently yields zeros on GPU: the\n    bound needs u < 1 strictly AND `u * a0` never rounding up to a0.  Both\n    hold.  tf.random.uniform returns [minval, maxval), so the largest float32\n    draw is 1 - 2^-24; and for a0 in [2^e, 2^(e+1)) the exact product\n    a0 - a0*2^-24 sits strictly between a0 - ulp and a0 - ulp/2, so\n    round-to-nearest gives a0 - ulp, not a0.  Verified over 1.2e7 draws\n    spanning a0 from 1e-6 to 1e6: zero occurrences.  Feeding u == 1.0 by hand\n    breaks this; the RNG cannot produce it.\n    """\n    cdf = tf.cumsum(propensities, axis=-1)\n    threshold = u * cdf[..., -1:]\n    # #{j : cdf[j] <= threshold} == smallest j with cdf[j] > threshold.\n    # Zero-propensity reactions leave cdf flat and are therefore unreachable,\n    # which is the correct direct-method behaviour.\n    index = tf.reduce_sum(tf.cast(cdf <= threshold, tf.int32), axis=-1)\n\n    if SSA_DEBUG_CHECKS:\n        num_reactions = tf.shape(propensities)[-1]\n        tf.debugging.assert_less(\n            index,\n            num_reactions,\n            message="direct_method_index: index out of range; either an "\n                    "all-zero propensity row reached the sampler (caller "\n                    "failed to mask absorbing states) or u == 1.0 was passed",\n        )\n        # The postcondition that actually matters, and which subsumes the\n        # range check: a zero-propensity reaction must never be selected.\n        tf.debugging.assert_positive(\n            tf.gather(\n                propensities,\n                tf.minimum(index, num_reactions - 1),\n                batch_dims=tf.rank(propensities) - 1,\n            ),\n            message="direct_method_index: selected a zero-propensity reaction",\n        )\n    return index\n\n\ndef direct_method_index_searchsorted(propensities, u):\n    """Identical semantics to direct_method_index; O(log R) instead of O(R).\n\n    For R <= ~32 the compare-and-count form above beats this on GPU (one fused\n    elementwise+reduce against a binary-search kernel launch).  Kept for large\n    reaction networks.  side="right" gives #{j : cdf[j] <= threshold}, matching\n    direct_method_index exactly.  Requires a rank-2 propensity tensor.\n    """\n    cdf = tf.cumsum(propensities, axis=-1)\n    threshold = u * cdf[..., -1:]\n    index = tf.squeeze(tf.searchsorted(cdf, threshold, side="right"), axis=-1)\n\n    if SSA_DEBUG_CHECKS:\n        tf.debugging.assert_less(\n            index,\n            tf.shape(propensities)[-1],\n            message="direct_method_index_searchsorted: index out of range",\n        )\n    return index\n\n\n# ------------------------------------------- direct-method PST estimators\n\n\ndef _hard_ssa_delta_core(propensities, S_matrix, u_reaction):\n    """Forward-only direct SSA reaction selection and stoichiometric gather.\n\n    Inputs are already detached by the caller.  This function is intentionally\n    gradient-agnostic: its sole job is\n\n        cdf -> threshold comparison -> integer reaction index -> S[index].\n\n    It is used in two forms below:\n      * ordinary TensorFlow graph execution (`PST-SSA`);\n      * an XLA-compiled subfunction (`PST-SSA-XLA`).\n\n    The outer differentiable simulator remains jit_compile=False in both cases.\n    """\n    hard_index = direct_method_index(propensities, u_reaction)\n    return tf.gather(S_matrix, hard_index)\n\n\n# Only this detached hard sampler is XLA-compiled.  The surrounding simulator,\n# loss, GradientTape, optimizer update, and PST custom VJP remain non-XLA.\nhard_ssa_delta_xla = tf.function(\n    _hard_ssa_delta_core,\n    jit_compile=True,\n    reduce_retracing=True,\n)\n\n\ndef _pst_affine_vjp(\n    propensities,\n    total_propensity,\n    S_matrix,\n    d_delta,\n):\n    """Analytic PST VJP of (a/a0) @ S with respect to raw propensities."""\n    # q[b,j] = <d_delta[b,:], S[j,:]>\n    q = tf.linalg.matmul(d_delta, S_matrix, transpose_b=True)\n\n    inv_a0 = tf.math.reciprocal(total_propensity)\n    probability = propensities * inv_a0\n    mean_q = tf.reduce_sum(\n        q * probability,\n        axis=-1,\n        keepdims=True,\n    )\n\n    return (q - mean_q) * inv_a0\n\n\n@tf.custom_gradient\ndef pst_ssa_delta_from_uniform(\n    propensities,\n    total_propensity,\n    S_matrix,\n    u_reaction,\n):\n    """Unfused direct-SSA hard jump + analytic PST affine VJP.\n\n    This is retained as the control arm for the XLA sampler experiment.\n    """\n    hard_prop = tf.stop_gradient(propensities)\n    hard_S = tf.stop_gradient(S_matrix)\n    hard_u = tf.stop_gradient(u_reaction)\n\n    delta_hard = _hard_ssa_delta_core(\n        hard_prop,\n        hard_S,\n        hard_u,\n    )\n\n    def grad(d_delta):\n        d_propensities = _pst_affine_vjp(\n            propensities,\n            total_propensity,\n            S_matrix,\n            d_delta,\n        )\n        return (\n            d_propensities,\n            tf.zeros_like(total_propensity),\n            None,\n            None,\n        )\n\n    return delta_hard, grad\n\n\n@tf.custom_gradient\ndef pst_ssa_delta_xla_from_uniform(\n    propensities,\n    total_propensity,\n    S_matrix,\n    u_reaction,\n):\n    """XLA-fused detached direct-SSA hard jump + the SAME analytic PST VJP.\n\n    The estimator is unchanged relative to PST-SSA.  Only the forward hard\n    sampler implementation changes:\n\n        PST-SSA:\n            _hard_ssa_delta_core(...)\n\n        PST-SSA-XLA:\n            hard_ssa_delta_xla(...)\n\n    This isolates whether fusing Cumsum/compare/reduce/gather removes the\n    TensorFlow direct-method sampler overhead seen for the oscillator.\n    """\n    hard_prop = tf.stop_gradient(propensities)\n    hard_S = tf.stop_gradient(S_matrix)\n    hard_u = tf.stop_gradient(u_reaction)\n\n    # Partitioned call to an XLA-compiled FORWARD-ONLY subgraph.\n    delta_hard = hard_ssa_delta_xla(\n        hard_prop,\n        hard_S,\n        hard_u,\n    )\n\n    def grad(d_delta):\n        d_propensities = _pst_affine_vjp(\n            propensities,\n            total_propensity,\n            S_matrix,\n            d_delta,\n        )\n        return (\n            d_propensities,\n            tf.zeros_like(total_propensity),\n            None,\n            None,\n        )\n\n    return delta_hard, grad\n\n\ndef clock_pst_delta_ssa(\n    propensities,\n    total_propensity,\n    S_matrix,\n    seed,\n    use_xla_sampler=False,\n):\n    """Stateless PST-SSA reaction jump, with optional XLA hard sampler."""\n    u = tf.random.stateless_uniform(\n        tf.concat([tf.shape(propensities)[:-1], [1]], axis=0),\n        seed=seed,\n        minval=0.0,\n        maxval=1.0,\n        dtype=propensities.dtype,\n    )\n\n    if use_xla_sampler:\n        return pst_ssa_delta_xla_from_uniform(\n            propensities,\n            total_propensity,\n            S_matrix,\n            u,\n        )\n\n    return pst_ssa_delta_from_uniform(\n        propensities,\n        total_propensity,\n        S_matrix,\n        u,\n    )\n\n\ndef ion_pst_delta_ssa(\n    propensities,\n    total_propensity,\n    S_matrix,\n    use_xla_sampler=False,\n):\n    """Stateful PST-SSA reaction jump, with optional XLA hard sampler."""\n    u = tf.random.uniform(\n        tf.concat([tf.shape(propensities)[:-1], [1]], axis=0),\n        minval=0.0,\n        maxval=1.0,\n        dtype=propensities.dtype,\n    )\n\n    if use_xla_sampler:\n        return pst_ssa_delta_xla_from_uniform(\n            propensities,\n            total_propensity,\n            S_matrix,\n            u,\n        )\n\n    return pst_ssa_delta_from_uniform(\n        propensities,\n        total_propensity,\n        S_matrix,\n        u,\n    )\n\n\ndef clock_simulate_ssa(batch_initial_states, num_steps, k, S_matrix, seed):\n    """Standard Gillespie direct method, same loop shape as the PST simulator.\n\n    Differences from the PST/GS-ST simulators, all intrinsic to the SSA rather\n    than incidental: the reaction is an integer index, not a one-hot row; the\n    state update is tf.gather(S, index), not one_hot @ S; and there is no\n    softmax, Gumbel, or stop_gradient, hence no gradient path from `k` to the\n    trajectory.\n\n    jit_compile stays False.  The reference generator elsewhere in this cell\n    uses jit_compile=True because it is never compared against a training\n    step; a JIT-compiled row here would not be comparable to the non-JIT\n    PST/GS-ST rows.\n    """\n    num_simulations = tf.shape(batch_initial_states)[0]\n    num_reactions = tf.shape(S_matrix)[0]\n    time_t = tf.zeros([num_simulations, 1], tf.float32)\n    states_ta = tf.TensorArray(tf.float32, size=num_steps + 1)\n    times_ta = tf.TensorArray(tf.float32, size=num_steps + 1)\n    states_ta = states_ta.write(0, batch_initial_states)\n    times_ta = times_ta.write(0, time_t)\n\n    t = tf.constant(0)\n    states = batch_initial_states\n\n    def cond(t, states, time_t, states_ta, times_ta, seed):\n        return t < num_steps\n\n    def body(t, states, time_t, states_ta, times_ta, seed):\n        seeds = tf.random.experimental.stateless_split(seed, 3)\n        next_seed, seed_time, seed_reaction = seeds[0], seeds[1], seeds[2]\n\n        prop = clock_propensity(states, k)\n        a0 = tf.reduce_sum(prop, axis=1, keepdims=True)\n        active = a0 > 0.0\n        safe_a0 = tf.where(active, a0, tf.ones_like(a0))\n\n        u_time = tf.random.stateless_uniform(\n            [num_simulations, 1],\n            seed=seed_time,\n            minval=1e-20,\n            maxval=1.0,\n            dtype=prop.dtype,\n        )\n        dt = tf.where(\n            active,\n            -tf.math.log(u_time) / safe_a0,\n            tf.zeros_like(u_time),\n        )\n\n        # Absorbing rows get a dummy positive propensity so the sampler\'s\n        # precondition holds; their state delta is zeroed below.\n        fallback = tf.one_hot(\n            tf.zeros([num_simulations], dtype=tf.int32),\n            depth=num_reactions,\n            dtype=prop.dtype,\n        )\n        sampling_prop = tf.where(active, prop, fallback)\n\n        u_reaction = tf.random.stateless_uniform(\n            [num_simulations, 1],\n            seed=seed_reaction,\n            minval=0.0,\n            maxval=1.0,\n            dtype=prop.dtype,\n        )\n        reaction_index = direct_method_index(sampling_prop, u_reaction)\n\n        delta = _apply_reaction(\n            reaction_index, S_matrix, num_reactions, prop.dtype\n        )\n        delta = tf.where(active, delta, tf.zeros_like(delta))\n\n        new_states = tf.maximum(states + delta, 0.0)\n        new_time = time_t + dt\n\n        states_ta = states_ta.write(t + 1, new_states)\n        times_ta = times_ta.write(t + 1, new_time)\n        return t + 1, new_states, new_time, states_ta, times_ta, next_seed\n\n    _, _, _, states_ta, times_ta, _ = tf.while_loop(\n        cond, body,\n        [t, states, time_t, states_ta, times_ta, seed],\n        shape_invariants=[\n            t.shape,\n            tf.TensorShape([None, 9]),\n            tf.TensorShape([None, 1]),\n            tf.TensorShape([]),\n            tf.TensorShape([]),\n            tf.TensorShape([2]),\n        ],\n    )\n    states = tf.transpose(states_ta.stack(), [1, 0, 2])\n    times = tf.transpose(times_ta.stack(), [1, 0, 2])\n    return states, times\n\n\n@tf.function(jit_compile=False, reduce_retracing=True)\ndef ion_simulate_ssa(batch_initial_states, k_open, k_close, k_inact):\n    """Direct method with the ion protocol\'s absorbing-state convention."""\n    num_sims = tf.shape(batch_initial_states)[0]\n    num_reactions = tf.shape(ION_S)[0]\n    time_t = tf.zeros([num_sims, 1], tf.float32)\n    states_ta = tf.TensorArray(tf.float32, size=ION_NUM_STEPS + 1)\n    times_ta = tf.TensorArray(tf.float32, size=ION_NUM_STEPS + 1)\n    states_ta = states_ta.write(0, batch_initial_states)\n    times_ta = times_ta.write(0, time_t)\n\n    states = batch_initial_states\n    step = tf.constant(0)\n\n    def cond(step, states, time_t, states_ta, times_ta):\n        return step < ION_NUM_STEPS\n\n    def body(step, states, time_t, states_ta, times_ta):\n        prop = ion_propensity(states, k_open, k_close, k_inact)\n        a0 = tf.reduce_sum(prop, axis=1, keepdims=True)\n\n        absorbing = a0 < 1e-10\n        safe_a0 = tf.where(absorbing, tf.ones_like(a0), a0)\n\n        u_time = tf.random.uniform(\n            [num_sims, 1], minval=1e-20, maxval=1.0, dtype=tf.float32\n        )\n        dt = -tf.math.log(u_time) / safe_a0\n        dt = tf.where(absorbing, tf.ones_like(dt) * 100000.0, dt)\n\n        # Direct SSA uses the true propensities. Zero-propensity reactions\n        # therefore have probability exactly zero.\n        fallback = tf.one_hot(\n            tf.zeros([num_sims], dtype=tf.int32),\n            depth=num_reactions,\n            dtype=prop.dtype,\n        )\n        sampling_prop = tf.where(absorbing, fallback, prop)\n\n        u_reaction = tf.random.uniform(\n            [num_sims, 1], minval=0.0, maxval=1.0, dtype=prop.dtype\n        )\n        reaction_index = direct_method_index(sampling_prop, u_reaction)\n\n        delta = _apply_reaction(\n            reaction_index, ION_S, num_reactions, prop.dtype\n        )\n        delta = tf.where(absorbing, tf.zeros_like(delta), delta)\n\n        states = tf.maximum(states + delta, 0.0)\n        time_t = time_t + dt\n\n        states_ta = states_ta.write(step + 1, states)\n        times_ta = times_ta.write(step + 1, time_t)\n        return step + 1, states, time_t, states_ta, times_ta\n\n    _, _, _, states_ta, times_ta = tf.while_loop(\n        cond, body,\n        [step, states, time_t, states_ta, times_ta],\n        shape_invariants=[\n            step.shape,\n            tf.TensorShape([None, 3]),\n            tf.TensorShape([None, 1]),\n            tf.TensorShape([]),\n            tf.TensorShape([]),\n        ],\n    )\n    states = tf.transpose(states_ta.stack(), [1, 0, 2])\n    times = tf.transpose(times_ta.stack(), [1, 0, 2])\n    return states, times\n\n\n# ------------------------------------------------ forward-only timing entries\n\ndef make_clock_ssa_forward():\n    """Simulation + rate loss.  No tape, no gradient, no optimizer.\n\n    There is deliberately no GradientTape here, and the reason is sharper than\n    "the SSA is non-differentiable".\n\n    The reaction-choice channel is genuinely dead: tf.gather by a sampled\n    integer index has no derivative, so d(states)/dk is exactly zero.  But the\n    waiting-time channel is NOT dead -- dt = -log(u)/a0 is an analytic\n    function of the propensity sum, so d(times)/dk is finite and nonzero.  The\n    clock loss is a rate, delta_state / delta_time, so it touches both.\n\n    A tape wrapped around the SSA therefore does not raise and does not return\n    None.  It returns a large, finite, entirely plausible-looking number that\n    is missing the whole reaction-selection term.  Timing that as a "training\n    step", or worse training on it, is a silent failure.  The only safe\n    options are to time the forward pass alone, as here, or to use one of the\n    straight-through estimators.  verify_ssa_gradient_channels() asserts both\n    halves of this.\n    """\n    k_const = tf.constant(CLOCK_K_TRUE, tf.float32)\n\n    @tf.function(\n        jit_compile=False,\n        reduce_retracing=True,\n        input_signature=[\n            tf.TensorSpec([CLOCK_BATCH_SIZE, CLOCK_M_TRAIN, 9], tf.float32),\n            tf.TensorSpec([CLOCK_BATCH_SIZE, CLOCK_SEGMENT_STEPS + 1, 9], tf.float32),\n            tf.TensorSpec([CLOCK_BATCH_SIZE, CLOCK_SEGMENT_STEPS + 1, 1], tf.float32),\n            tf.TensorSpec([2], tf.int32),\n        ],\n    )\n    def forward_step(batch_initial_states, batch_target_states,\n                     batch_target_times, seed2):\n        super_batch_initial_states = tf.repeat(\n            batch_initial_states[:, 0, :], repeats=CLOCK_M_TRAIN, axis=0\n        )\n        super_states, super_times = clock_simulate_ssa(\n            super_batch_initial_states,\n            tf.constant(CLOCK_SEGMENT_STEPS, tf.int32),\n            k_const,\n            CLOCK_S,\n            seed2,\n        )\n        sim_states = tf.reshape(\n            super_states,\n            [CLOCK_BATCH_SIZE, CLOCK_M_TRAIN, CLOCK_SEGMENT_STEPS + 1, 9],\n        )\n        sim_times = tf.reshape(\n            super_times,\n            [CLOCK_BATCH_SIZE, CLOCK_M_TRAIN, CLOCK_SEGMENT_STEPS + 1, 1],\n        )\n        mean_states = tf.reduce_mean(sim_states, axis=1)\n        mean_times = tf.reduce_mean(sim_times, axis=1)\n        loss = clock_compute_loss(\n            mean_states, batch_target_states, mean_times, batch_target_times\n        )\n        return tf.stack([loss])\n\n    def reset():\n        # No trainable variables and no optimizer slots exist for this entry.\n        return None\n\n    return forward_step, reset\n\n\ndef make_ion_ssa_forward(t_target, n_open_target):\n    """Ion-channel simulation + loss, forward only.  See make_clock_ssa_forward."""\n    initial_state = tf.constant([[2.0, 0.0, 0.0]], tf.float32)\n    k_open = tf.constant(0.25, tf.float32)\n    k_close = tf.constant(0.25, tf.float32)\n    k_inact = tf.constant(0.25, tf.float32)\n\n    @tf.function(jit_compile=False, reduce_retracing=True)\n    def forward_step_graph():\n        super_batch = tf.repeat(initial_state, repeats=ION_M_SIMS, axis=0)\n        sim_states, sim_times = ion_simulate_ssa(\n            super_batch, k_open, k_close, k_inact\n        )\n        n_open_sim = sim_states[:, :, 1]\n        model_bin_avg = ion_bin_integrated_readout(\n            sim_times, n_open_sim, t_target\n        )\n        model_bin_avg = tf.reshape(\n            model_bin_avg, [1, ION_M_SIMS, tf.shape(t_target)[0] - 1]\n        )\n        model_mean = tf.reduce_mean(model_bin_avg, axis=[0, 1])\n        experimental_bin_avg = 0.5 * (n_open_target[:-1] + n_open_target[1:])\n        return tf.reduce_mean(tf.square(model_mean - experimental_bin_avg))\n\n    def forward_step(_seed_unused):\n        # Stateful RNG, matching the ion protocol; the seed argument is unused\n        # for the same reason it is unused in make_ion_trainer.\n        return np.array([float(forward_step_graph().numpy())], dtype=np.float64)\n\n    def reset():\n        return None\n\n    return forward_step, reset\n\n\n# --------------------------------------------------------------- diagnostics\n\ndef verify_ssa_sampler(num_draws=400_000, seed=(7, 11)):\n    """Assert the direct-method sampler reproduces prop / a0, including zeros."""\n    propensity_row = tf.constant(\n        [[3.0, 0.0, 1.0, 6.0, 0.0, 10.0]], tf.float32\n    )\n    num_reactions = int(propensity_row.shape[-1])\n    batch = tf.tile(propensity_row, [num_draws, 1])\n    u = tf.random.stateless_uniform([num_draws, 1], seed=seed, dtype=tf.float32)\n    index = direct_method_index(batch, u)\n\n    assert int(tf.reduce_max(index)) < num_reactions, "index out of range"\n    assert int(tf.reduce_min(index)) >= 0, "negative index"\n\n    counts = tf.math.bincount(index, minlength=num_reactions)\n    empirical = counts.numpy() / num_draws\n    expected = (propensity_row / tf.reduce_sum(propensity_row)).numpy()[0]\n\n    zero_mask = expected == 0.0\n    assert (empirical[zero_mask] == 0.0).all(), (\n        "zero-propensity reaction was selected"\n    )\n    max_error = float(np.max(np.abs(empirical - expected)))\n    tolerance = 4.0 / np.sqrt(num_draws)\n    assert max_error < tolerance, (\n        f"sampler deviates from prop/a0: max |error| = {max_error:.5f} "\n        f"> {tolerance:.5f}"\n    )\n    print(f"  verify_ssa_sampler: OK (max |error| = {max_error:.5f})")\n\n\ndef verify_ssa_gradient_channels():\n    """Pin down which SSA channel carries a gradient and which does not.\n\n    Asserts the two facts that make the forward-only design mandatory:\n      * d(states)/dk == 0  -- the reaction choice is a hard integer index;\n      * d(times)/dk  != 0  -- the waiting time is reparameterised in a0.\n\n    The second is the trap: a GradientTape over the SSA returns a finite,\n    plausible, WRONG gradient for any loss that reads the clock, rather than\n    failing.\n    """\n    log_k = tf.Variable(tf.math.log(tf.constant(CLOCK_K_TRUE, tf.float32)))\n    initial = tf.constant([[1., 0., 1., 0., 0., 0., 0., 0., 0.]], tf.float32)\n    seed = tf.constant([1, 2], tf.int32)\n\n    with tf.GradientTape() as tape:\n        states, _ = clock_simulate_ssa(\n            initial, tf.constant(8, tf.int32), tf.exp(log_k), CLOCK_S, seed\n        )\n        state_objective = tf.reduce_sum(states)\n    state_gradient = tape.gradient(state_objective, log_k)\n    # TensorFlow materialises structural zeros rather than None through\n    # while_loop backprop, so accept either but require exact zero.\n    if state_gradient is not None:\n        leaked = float(tf.reduce_max(tf.abs(state_gradient)))\n        assert leaked == 0.0, (\n            f"the reaction-selection path leaked a gradient; max |g| = {leaked}"\n        )\n\n    with tf.GradientTape() as tape:\n        _, times = clock_simulate_ssa(\n            initial, tf.constant(8, tf.int32), tf.exp(log_k), CLOCK_S, seed\n        )\n        time_objective = tf.reduce_sum(times)\n    time_gradient = tape.gradient(time_objective, log_k)\n    assert time_gradient is not None, "the waiting-time path lost its gradient"\n    time_magnitude = float(tf.reduce_max(tf.abs(time_gradient)))\n    assert time_magnitude > 0.0, (\n        "the waiting-time path lost its gradient; the partial-gradient hazard "\n        "documented in make_clock_ssa_forward no longer applies and that "\n        "docstring must be corrected"\n    )\n\n    print(\n        "  verify_ssa_gradient_channels: OK "\n        f"(d states/dk == 0, d times/dk = {time_magnitude:.3e} != 0)"\n    )\n\n\n\n\ndef verify_pst_direct_backward_matches():\n    """Verify optimized affine custom VJP equals the standard PST VJP."""\n    prop_ref = tf.constant([[3.0, 1.0, 6.0, 10.0]], tf.float32)\n    prop_opt = tf.constant([[3.0, 1.0, 6.0, 10.0]], tf.float32)\n\n    S_test = tf.constant(\n        [\n            [1.0, 0.0, -1.0],\n            [0.0, 1.0,  1.0],\n            [-1.0, 0.0, 2.0],\n            [0.0, -1.0, 0.5],\n        ],\n        tf.float32,\n    )\n    probe = tf.constant([[0.7, -1.3, 2.1]], tf.float32)\n\n    with tf.GradientTape() as tape_ref:\n        tape_ref.watch(prop_ref)\n        pi_ref = tf.nn.softmax(tf.math.log(prop_ref), axis=-1)\n        delta_ref = tf.matmul(pi_ref, S_test)\n        scalar_ref = tf.reduce_sum(delta_ref * probe)\n    grad_ref = tape_ref.gradient(scalar_ref, prop_ref)\n\n    with tf.GradientTape() as tape_opt:\n        tape_opt.watch(prop_opt)\n        a0 = tf.reduce_sum(prop_opt, axis=-1, keepdims=True)\n        delta_opt = pst_ssa_delta_from_uniform(\n            prop_opt,\n            a0,\n            S_test,\n            tf.constant([[0.37]], tf.float32),\n        )\n        scalar_opt = tf.reduce_sum(delta_opt * probe)\n    grad_opt = tape_opt.gradient(scalar_opt, prop_opt)\n\n    np.testing.assert_allclose(\n        grad_ref.numpy(),\n        grad_opt.numpy(),\n        rtol=2e-6,\n        atol=2e-7,\n        err_msg=(\n            "Optimized PST-SSA affine custom VJP does not match "\n            "standard PST d/d(a)"\n        ),\n    )\n\n    print(\n        "  verify_pst_direct_backward_matches: OK "\n        "(optimized affine VJP == standard PST d/d(a))"\n    )\n\n\n\n\ndef verify_xla_hard_sampler_matches_unfused():\n    """Bitwise-equivalent hard SSA jump for identical propensities and U."""\n    print("\\nPST-SSA XLA hard-sampler equivalence check")\n    print("------------------------------------------")\n\n    prop = tf.constant(\n        [\n            [3.0, 1.0, 6.0, 10.0],\n            [2.0, 5.0, 1.0,  2.0],\n            [0.0, 4.0, 0.0,  6.0],\n        ],\n        tf.float32,\n    )\n    S_test = tf.constant(\n        [\n            [1.0, 0.0, -1.0],\n            [0.0, 1.0,  1.0],\n            [-1.0, 0.0, 2.0],\n            [0.0, -1.0, 0.5],\n        ],\n        tf.float32,\n    )\n    u = tf.constant([[0.13], [0.71], [0.39]], tf.float32)\n\n    unfused = _hard_ssa_delta_core(prop, S_test, u)\n    fused = hard_ssa_delta_xla(prop, S_test, u)\n\n    tf.debugging.assert_equal(unfused, fused)\n\n    print("  hard SSA delta: exact match")\n    print("  XLA hard-sampler call executed successfully.")\n\n\ndef verify_xla_pst_vjp_matches_unfused():\n    """XLA and unfused PST-SSA variants must expose the same custom VJP."""\n    print("\\nPST-SSA XLA custom-VJP equivalence check")\n    print("----------------------------------------")\n\n    prop_a = tf.constant([[3.0, 1.0, 6.0, 10.0]], tf.float32)\n    prop_b = tf.constant([[3.0, 1.0, 6.0, 10.0]], tf.float32)\n\n    S_test = tf.constant(\n        [\n            [1.0, 0.0, -1.0],\n            [0.0, 1.0,  1.0],\n            [-1.0, 0.0, 2.0],\n            [0.0, -1.0, 0.5],\n        ],\n        tf.float32,\n    )\n    u = tf.constant([[0.37]], tf.float32)\n    probe = tf.constant([[0.7, -1.3, 2.1]], tf.float32)\n\n    with tf.GradientTape() as tape_a:\n        tape_a.watch(prop_a)\n        a0 = tf.reduce_sum(prop_a, axis=-1, keepdims=True)\n        delta_a = pst_ssa_delta_from_uniform(\n            prop_a, a0, S_test, u\n        )\n        scalar_a = tf.reduce_sum(delta_a * probe)\n    grad_a = tape_a.gradient(scalar_a, prop_a)\n\n    with tf.GradientTape() as tape_b:\n        tape_b.watch(prop_b)\n        a0 = tf.reduce_sum(prop_b, axis=-1, keepdims=True)\n        delta_b = pst_ssa_delta_xla_from_uniform(\n            prop_b, a0, S_test, u\n        )\n        scalar_b = tf.reduce_sum(delta_b * probe)\n    grad_b = tape_b.gradient(scalar_b, prop_b)\n\n    np.testing.assert_allclose(\n        grad_a.numpy(),\n        grad_b.numpy(),\n        rtol=1e-6,\n        atol=1e-7,\n    )\n\n    print("  custom PST VJP: exact numerical match within tolerance.")\n\n\ndef verify_streamlined_pst_ssa_estimator_graph():\n    """Confirm optimized PST-SSA avoids unnecessary reaction-selection ops."""\n    print("\\nOptimized PST-SSA affine-op graph check")\n    print("---------------------------------------")\n\n    prop = tf.constant(\n        [[3.0, 1.0, 6.0, 10.0],\n         [2.0, 5.0, 1.0, 2.0]],\n        tf.float32,\n    )\n    a0 = tf.reduce_sum(prop, axis=-1, keepdims=True)\n    S_test = tf.constant(\n        [\n            [1.0, 0.0, -1.0],\n            [0.0, 1.0,  1.0],\n            [-1.0, 0.0, 2.0],\n            [0.0, -1.0, 0.5],\n        ],\n        tf.float32,\n    )\n    u = tf.constant([[0.2], [0.8]], tf.float32)\n\n    @tf.function(jit_compile=False)\n    def pst_ssa_forward(a, total, S, draw):\n        return pst_ssa_delta_from_uniform(a, total, S, draw)\n\n    concrete = pst_ssa_forward.get_concrete_function(prop, a0, S_test, u)\n    counts = graph_op_counts(concrete)\n\n    forbidden = {\n        "Softmax": counts.get("Softmax", 0),\n        "Log": counts.get("Log", 0),\n        "OneHot": counts.get("OneHot", 0),\n    }\n\n    for name, count in forbidden.items():\n        print(f"  {name:<8} ops: {count}")\n    print(f"  Cumsum   ops: {counts.get(\'Cumsum\', 0)}")\n    print(f"  GatherV2 ops: {counts.get(\'GatherV2\', 0)}")\n\n    if any(forbidden.values()):\n        raise AssertionError(\n            "Optimized PST-SSA unexpectedly contains forbidden forward ops: "\n            f"{forbidden}"\n        )\n\n    print(\n        "  optimized forward confirmed: raw-propensity CDF + hard S[j] gather."\n    )\n\n\n\ndef verify_optimized_pst_ssa_gradient_graph():\n    """Inspect a forward+backward wrapper around the custom affine PST op."""\n    print("\\nOptimized PST-SSA forward+backward graph check")\n    print("-----------------------------------------------")\n\n    prop = tf.constant(\n        [[3.0, 1.0, 6.0, 10.0],\n         [2.0, 5.0, 1.0, 2.0]],\n        tf.float32,\n    )\n    S_test = tf.constant(\n        [\n            [1.0, 0.0, -1.0],\n            [0.0, 1.0,  1.0],\n            [-1.0, 0.0, 2.0],\n            [0.0, -1.0, 0.5],\n        ],\n        tf.float32,\n    )\n    u = tf.constant([[0.2], [0.8]], tf.float32)\n    probe = tf.constant(\n        [[0.7, -1.3, 2.1], [1.1, 0.4, -0.8]],\n        tf.float32,\n    )\n\n    @tf.function(jit_compile=False)\n    def value_and_grad(a, S, draw, adjoint):\n        with tf.GradientTape() as tape:\n            tape.watch(a)\n            total = tf.reduce_sum(a, axis=-1, keepdims=True)\n            delta = pst_ssa_delta_from_uniform(a, total, S, draw)\n            scalar = tf.reduce_sum(delta * adjoint)\n        return scalar, tape.gradient(scalar, a)\n\n    concrete = value_and_grad.get_concrete_function(\n        prop, S_test, u, probe\n    )\n    counts = graph_op_counts(concrete)\n\n    print(f"  Softmax ops: {counts.get(\'Softmax\', 0)}")\n    print(f"  Log ops:     {counts.get(\'Log\', 0)}")\n    print(f"  Cumsum ops:  {counts.get(\'Cumsum\', 0)}")\n    print(f"  MatMul ops:  {counts.get(\'MatMul\', 0)}")\n\n    if counts.get("Softmax", 0) or counts.get("Log", 0):\n        raise AssertionError(\n            "Optimized PST-SSA gradient graph contains Softmax/Log."\n        )\n\n    print(\n        "  custom analytic propensity-space VJP is active; "\n        "the hard CDF sampler is forward-only."\n    )\n\n\ndef run_ssa_diagnostics():\n    """SSA/direct-method correctness checks; never inside timed intervals."""\n    print("\\nSSA diagnostics")\n    print("---------------")\n    verify_ssa_sampler(num_draws=SSA_DIAGNOSTIC_DRAWS)\n    verify_ssa_gradient_channels()\n    if INCLUDE_PST_DIRECT:\n        verify_streamlined_pst_ssa_estimator_graph()\n        verify_optimized_pst_ssa_gradient_graph()\n        verify_pst_direct_backward_matches()\n    if INCLUDE_PST_XLA:\n        verify_xla_hard_sampler_matches_unfused()\n        verify_xla_pst_vjp_matches_unfused()\n\n\n# =============================================================================\n# COST-PARITY VERIFICATION\n# =============================================================================\n#\n# Timing alone cannot distinguish an estimator-level cost difference from an\n# implementation asymmetry in the surrounding loop.  These checks trace the\n# two reaction-selection rules and compare their op-type counts directly, so\n# the cost claim is verified structurally before any clock is read.\n\ndef graph_op_counts(concrete_fn):\n    """Op-type histogram of a traced graph, including nested function bodies.\n\n    A tf.while_loop places its body in a separate FunctionDef, so counting\n    only the top-level nodes would miss the per-step work entirely.  The\n    serialized GraphDef carries those bodies in its `library` field.\n    """\n    graph_def = concrete_fn.graph.as_graph_def()\n    counts = {}\n    for node in graph_def.node:\n        counts[node.op] = counts.get(node.op, 0) + 1\n    for func in graph_def.library.function:\n        for node in func.node_def:\n            counts[node.op] = counts.get(node.op, 0) + 1\n    return counts\n\n\ndef diff_op_counts(counts_a, counts_b):\n    """Op types whose counts differ, as {op_type: (count_a, count_b)}."""\n    keys = set(counts_a) | set(counts_b)\n    return {\n        k: (counts_a.get(k, 0), counts_b.get(k, 0))\n        for k in sorted(keys)\n        if counts_a.get(k, 0) != counts_b.get(k, 0)\n    }\n\n\n\ndef check_estimator_parity(num_rows=1024, num_reactions=16, tolerance=6):\n    """Structural check for the matched PST-GM / GSST-GM estimator pair.\n\n    Op counts are used only to catch accidental graph asymmetry. They are not\n    interpreted as runtime cost.\n    """\n    print("\\nMatched-sampler estimator graph structure")\n    print("----------------------------------------")\n\n    logits = tf.random.uniform(\n        [num_rows, num_reactions], -3.0, 3.0, tf.float32\n    )\n    seed = tf.constant([1, 2], tf.int32)\n    tau = tf.constant(3e-6, tf.float32)\n\n    @tf.function(jit_compile=False)\n    def pst_fn(x, s):\n        return clock_pst_gumbel_estimator(x, s)\n\n    @tf.function(jit_compile=False)\n    def gsst_fn(x, t, s):\n        return clock_gsst_gumbel_estimator(x, t, s)\n\n    pst_counts = graph_op_counts(\n        pst_fn.get_concrete_function(logits, seed)\n    )\n    gsst_counts = graph_op_counts(\n        gsst_fn.get_concrete_function(logits, tau, seed)\n    )\n\n    n_pst = sum(pst_counts.values())\n    n_gsst = sum(gsst_counts.values())\n    print(f"  {PST_GM_NAME:<8} total ops: {n_pst}")\n    print(f"  {GSST_GM_NAME:<8} total ops: {n_gsst}")\n\n    differences = diff_op_counts(pst_counts, gsst_counts)\n    if differences:\n        print(f"  differing op types ({PST_GM_NAME}, {GSST_GM_NAME}):")\n        for op_type, (a, b) in differences.items():\n            print(f"    {op_type:<24} {a:>4}  {b:>4}")\n    else:\n        print("  op-type histograms are identical")\n\n    if abs(n_gsst - n_pst) > tolerance:\n        raise AssertionError(\n            "Matched-sampler estimator graphs differ by more than expected. "\n            f"{PST_GM_NAME}={n_pst}, {GSST_GM_NAME}={n_gsst}; "\n            f"differences={differences}"\n        )\n\n    print("  structural path check passed; timing determines cost.")\n    return pst_counts, gsst_counts\n\n\n\ndef check_clock_graph_parity(pst_gm_step, gsst_gm_step):\n    """Structural check of complete matched-sampler clock training graphs.\n\n    This checks implementation-path similarity only; op counts are not a\n    performance measurement.\n    """\n    print("\\nMatched-sampler oscillator training-step graph structure")\n    print("--------------------------------------------------------")\n\n    pst_counts = graph_op_counts(pst_gm_step.get_concrete_function())\n    gsst_counts = graph_op_counts(gsst_gm_step.get_concrete_function())\n\n    n_pst = sum(pst_counts.values())\n    n_gsst = sum(gsst_counts.values())\n    print(f"  {PST_GM_NAME:<8} total ops: {n_pst}")\n    print(f"  {GSST_GM_NAME:<8} total ops: {n_gsst}")\n\n    differences = diff_op_counts(pst_counts, gsst_counts)\n    if differences:\n        print(f"  differing op types ({PST_GM_NAME}, {GSST_GM_NAME}):")\n        for op_type, (a, b) in differences.items():\n            print(f"    {op_type:<24} {a:>4}  {b:>4}")\n    else:\n        print("  op-type histograms are identical")\n    return pst_counts, gsst_counts\n\n\n# =============================================================================\n# PAIRED TIMING STATISTICS\n# =============================================================================\n\ndef paired_timing_stats(model, num_name, num_samples, den_name, den_samples,\n                        n_boot=20000, seed=0):\n    """Paired summary of the per-repeat timings for one pair of methods.\n\n    `time_paired` alternates execution order within a repeat but stores by\n    method, so entry r of each list comes from the same repeat and the lists\n    are index-aligned. Paired bootstrap resampling preserves this alignment\n    when estimating uncertainty in the ratio of median times.\n    """\n    numerator_samples = num_samples\n    denominator_samples = den_samples\n    if len(numerator_samples) != len(denominator_samples):\n        raise ValueError(\n            f"{model} {num_name}/{den_name}: unpaired timing samples, "\n            f"{len(numerator_samples)} vs {len(denominator_samples)}"\n        )\n    n = len(numerator_samples)\n    if n < 2:\n        raise ValueError(f"{model}: paired statistics need >= 2 repeats, got {n}")\n\n    diffs = [a - b for a, b in zip(numerator_samples, denominator_samples)]\n    mean_diff = statistics.mean(diffs)\n    sd_diff = statistics.stdev(diffs)\n    if sd_diff <= 0.0:\n        raise ValueError(\n            f"{model}: zero variance in paired differences. Identical timings "\n            "on every repeat means the timer is not resolving the calls."\n        )\n    t_stat = mean_diff / (sd_diff / math.sqrt(n))\n    numerator_slower = sum(1 for d in diffs if d > 0.0)\n\n    denominator_median = statistics.median(denominator_samples)\n    if denominator_median <= 0.0:\n        raise ValueError(\n            f"{model}: non-positive {den_name} median {denominator_median}"\n        )\n    median_ratio = (\n        statistics.median(numerator_samples) / denominator_median\n    )\n\n    rng = random.Random(seed)\n    ratios = []\n    for _ in range(n_boot):\n        idx = [rng.randrange(n) for _ in range(n)]\n        den = statistics.median([denominator_samples[i] for i in idx])\n        if den <= 0.0:\n            raise ValueError(f"{model}: non-positive median in bootstrap resample")\n        ratios.append(statistics.median([numerator_samples[i] for i in idx]) / den)\n    ratios.sort()\n    lo = ratios[int(0.025 * n_boot)]\n    hi = ratios[int(0.975 * n_boot) - 1]\n    assert lo <= hi, f"{model}: bootstrap interval inverted ({lo}, {hi})"\n\n    return PairedStats(\n        model=model,\n        numerator=num_name,\n        denominator=den_name,\n        repeats=n,\n        median_ratio=median_ratio,\n        ratio_ci_low=lo,\n        ratio_ci_high=hi,\n        mean_diff_ms=mean_diff,\n        sd_diff_ms=sd_diff,\n        t_statistic=t_stat,\n        degrees_of_freedom=n - 1,\n        numerator_slower_count=numerator_slower,\n    )\n\n\n# Paired comparisons answer distinct questions:\n#   PST-SSA-XLA/PST-SSA: direct measurement of hard-sampler XLA fusion.\n#   PST-SSA-XLA/PST-GM: fused direct SSA vs explicit Gumbel-Max, same PST rule.\n#   PST-GM/GSST-GM: backward-rule cost with hard Gumbel-Max sampler matched.\n#   PST-cat/GSST-GM and PST-SSA/GSST-GM: end-to-end comparisons.\n# SSA-fwd is deliberately absent because it is not a training step.\nPAIRED_COMPARISONS = [\n    # Hard-sampler implementation comparison: estimator and direct-method\n    # semantics are identical; only the detached hard sampler is XLA-fused.\n    (PST_XLA_NAME, PST_DIRECT_NAME),\n\n    # Compare the XLA direct sampler against the other PST implementations.\n    (PST_XLA_NAME, PST_CAT_NAME),\n    (PST_XLA_NAME, PST_GM_NAME),\n\n    # Existing controls.\n    (PST_CAT_NAME, PST_DIRECT_NAME),\n    (PST_CAT_NAME, PST_GM_NAME),\n    (PST_DIRECT_NAME, PST_GM_NAME),\n\n    # Same explicit Gumbel-Max hard sampler, different backward rules.\n    (PST_GM_NAME, GSST_GM_NAME),\n\n    # End-to-end comparisons against GS-ST.\n    (PST_CAT_NAME, GSST_GM_NAME),\n    (PST_DIRECT_NAME, GSST_GM_NAME),\n    (PST_XLA_NAME, GSST_GM_NAME),\n]\n\n\n\ndef paired_stats_for(results):\n    """Paired statistics for every configured comparison present in results."""\n    stats = []\n    for model in sorted({r.model for r in results}):\n        group = {r.method: r for r in results if r.model == model}\n        required = {PST_CAT_NAME, PST_GM_NAME, GSST_GM_NAME}\n        if INCLUDE_PST_DIRECT:\n            required.add(PST_DIRECT_NAME)\n        if INCLUDE_PST_XLA:\n            required.add(PST_XLA_NAME)\n        if not required.issubset(group):\n            raise ValueError(\n                f"{model}: expected {sorted(required)}, got {sorted(group)}"\n            )\n        for num_name, den_name in PAIRED_COMPARISONS:\n            if num_name not in group or den_name not in group:\n                continue\n            stats.append(\n                paired_timing_stats(\n                    model,\n                    num_name,\n                    group[num_name].samples_ms,\n                    den_name,\n                    group[den_name].samples_ms,\n                )\n            )\n    return stats\n\n\n# =============================================================================\n# REPORTING\n# =============================================================================\n\ndef print_results(results):\n    print("\\nTiming results")\n    print("--------------")\n    header = (\n        f"{\'model\':<8} {\'method\':<9} {\'median ms\':>12} "\n        f"{\'mean±sd ms\':>20} {\'ns/event-eq\':>14} {\'M events/s\':>12}"\n    )\n    print(header)\n    print("-" * len(header))\n    for r in results:\n        print(\n            f"{r.model:<8} {r.method:<9} {r.median_ms:12.3f} "\n            f"{r.mean_ms:9.3f}±{r.std_ms:<9.3f} "\n            f"{r.ns_per_trajectory_event:14.3f} "\n            f"{r.million_trajectory_events_per_s:12.3f}"\n        )\n\n    print("\\nPaired per-step cost, training steps only")\n    print("----------------------------------------")\n    print(\n        "Ratio is numerator/denominator; below 1 favours the numerator.  The\\n"\n        "bootstrap interval is over paired repeats; an interval containing 1.0\\n"\n        "means the two are not separated at this repeat count."\n    )\n    for stats in paired_stats_for(results):\n        label = f"{stats.model} {stats.numerator}/{stats.denominator}"\n        print(\n            f"\\n{label}: median ratio {stats.median_ratio:.4f}x "\n            f"[95% CI {stats.ratio_ci_low:.3f}-{stats.ratio_ci_high:.3f}], "\n            f"n={stats.repeats}"\n        )\n        print(\n            f"  paired diff {stats.mean_diff_ms:+.2f} ms "\n            f"(sd {stats.sd_diff_ms:.2f}), t={stats.t_statistic:+.2f} "\n            f"on {stats.degrees_of_freedom} df, {stats.numerator} slower in "\n            f"{stats.numerator_slower_count}/{stats.repeats} repeats"\n        )\n        if stats.ratio_ci_low <= 1.0 <= stats.ratio_ci_high:\n            print("  -> consistent with equal per-step cost.")\n        else:\n            slower = (\n                stats.numerator if stats.median_ratio > 1.0\n                else stats.denominator\n            )\n            print(\n                f"  -> {slower} is measurably more expensive per step. Check "\n                "graph parity above before\\n     reading this as an "\n                "estimator-level rather than an implementation difference."\n            )\n\n    ssa_models = sorted({r.model for r in results if r.method == SSA_METHOD_NAME})\n    if ssa_models:\n        print("\\nDifferentiable-step cost relative to the exact SSA forward")\n        print("---------------------------------------------------------")\n        print(\n            f"\'{SSA_METHOD_NAME}\' is forward simulation + loss ONLY: no tape, no\\n"\n            "backward pass, no optimizer update.  It is a lower bound on the\\n"\n            "step, not a comparable training step, so these are ratios without\\n"\n            "paired statistics."\n        )\n        for model in ssa_models:\n            group = {r.method: r for r in results if r.model == model}\n            base = group[SSA_METHOD_NAME].median_ms\n            if base <= 0.0:\n                raise ValueError(f"{model}: non-positive SSA forward median {base}")\n            for m in (\n                PST_CAT_NAME,\n                PST_DIRECT_NAME,\n                PST_XLA_NAME,\n                PST_GM_NAME,\n                GSST_GM_NAME,\n            ):\n                if m in group:\n                    print(\n                        f"  {model:<6} {m:<8}: "\n                        f"{group[m].median_ms / base:6.3f}x SSA forward"\n                    )\n\n\ndef save_results(results):\n    import csv\n\n    fields = [\n        "model", "method", "trajectories", "ssa_steps", "trajectory_events",\n        "repeats", "median_ms", "mean_ms", "std_ms", "min_ms", "max_ms",\n        "ns_per_trajectory_event", "million_trajectory_events_per_s",\n    ]\n    with OUTPUT_CSV.open("w", newline="") as f:\n        w = csv.DictWriter(f, fieldnames=fields)\n        w.writeheader()\n        for result in results:\n            row = asdict(result)\n            row.pop("samples_ms")\n            w.writerow(row)\n\n    payload = {\n        "environment": environment_metadata(),\n        "configuration": {\n                "benchmark_profile": BENCHMARK_PROFILE,\n            "benchmark_device": BENCHMARK_DEVICE,\n            "gpu_id_requested": GPU_ID,\n            "cpu_threads_requested": CPU_THREADS,\n            "jit_compile_training": False,\n            "reference_jit_compile": REFERENCE_JIT_COMPILE,\n            "ion_readout_jit_compile": ION_READOUT_JIT,\n            "warmup": WARMUP,\n            "repeats": REPEATS,\n            "ion_M_sims": ION_M_SIMS,\n            "ion_steps": ION_NUM_STEPS,\n            "ion_timing_epoch": ION_TIMING_EPOCH,\n            "clock_batch_size": CLOCK_BATCH_SIZE,\n            "clock_M_train": CLOCK_M_TRAIN,\n            "clock_steps": CLOCK_SEGMENT_STEPS,\n            "clock_reference_file": CLOCK_REFERENCE_FILE.name,\n            "clock_reference_steps": CLOCK_REFERENCE_STEPS,\n            "clock_burn_in": CLOCK_BURN_IN,\n            "ssa_diagnostic_draws": SSA_DIAGNOSTIC_DRAWS,\n            "ion_data_file": ION_DATA_FILE.name,\n            "clock_batch_selection_timed": False,\n            "optimizer_update_timed": True,\n            "primary_methods": [\n                PST_CAT_NAME,\n                PST_DIRECT_NAME,\n                PST_XLA_NAME,\n                PST_GM_NAME,\n                GSST_GM_NAME,\n            ],\n            "include_ssa_baseline": INCLUDE_SSA_BASELINE,\n            "include_pst_direct": INCLUDE_PST_DIRECT,\n            "include_pst_xla": INCLUDE_PST_XLA,\n            "ssa_onehot_state_update": SSA_ONEHOT_STATE_UPDATE,\n            "ssa_debug_checks": SSA_DEBUG_CHECKS,\n        },\n        "results": [asdict(r) for r in results],\n        "paired_stats": [asdict(s) for s in paired_stats_for(results)],\n    }\n    OUTPUT_JSON.write_text(json.dumps(payload, indent=2))\n    print(f"\\nSaved {OUTPUT_CSV}")\n    print(f"Saved {OUTPUT_JSON.name}")\n\n\ndef run_benchmark():\n    print_environment()\n    print("\\nBenchmark workload")\n    print("------------------")\n    print(f"profile: {BENCHMARK_PROFILE}")\n    print(f"ion:   {ION_M_SIMS:,} trajectories x {ION_NUM_STEPS} event slots")\n    print(\n        f"clock: {CLOCK_BATCH_SIZE:,} starts x {CLOCK_M_TRAIN} trajectories/start "\n        f"x {CLOCK_SEGMENT_STEPS} events"\n    )\n    print(f"warm-up calls/method: {WARMUP}; timed repeats/method: {REPEATS}")\n\n    check_estimator_parity()\n    verify_clock_matched_gumbel_forward()\n\n    if INCLUDE_SSA_BASELINE or INCLUDE_PST_DIRECT or INCLUDE_PST_XLA:\n        run_ssa_diagnostics()\n\n    results = []\n\n    if RUN_ION:\n        results.extend(benchmark_ion())\n\n    if RUN_CLOCK:\n        results.extend(benchmark_clock())\n\n    print_results(results)\n    save_results(results)\n    return results\n\nif __name__ == "__main__":\n    run_benchmark()\n'

WORKER_PATH = Path("pst_gsst_gpu_cpu_benchmark_worker.py")
WORKER_PATH.write_text(WORKER_SOURCE)
print("Wrote worker:", WORKER_PATH.name)


In [ ]:
# =============================================================================
# BENCHMARK CONDITIONS
# =============================================================================

conditions = []

if RUN_GPU_PRIMARY:
    conditions.append({
        "label": "gpu_primary",
        "device": "gpu",
        "cpu_threads": "auto",
        "ion_m_sims": 262_144,
        "clock_num_start_points": 262_144,
        "clock_reference_steps": 2_400_000,
        "clock_burn_in": 300_000,
        "clock_batch_size": 8_192,
        "warmup": GPU_WARMUP,
        "repeats": GPU_REPEATS,
        "ssa_diagnostic_draws": 400_000,
        "clock_reference_file": "reference_trajectory.npz",
        # Outside timing; keeps full-reference preparation fast.
        "reference_jit": True,
        # Shared by all ion rows, matching the established full-GPU benchmark.
        "ion_readout_jit": True,
    })

if RUN_CPU_PRIMARY:
    conditions.append({
        "label": "cpu_primary",
        "device": "cpu",
        "cpu_threads": "auto",
        "ion_m_sims": 32_768,
        "clock_num_start_points": 8_192,
        "clock_reference_steps": 600_000,
        "clock_burn_in": 75_000,
        "clock_batch_size": 1_024,
        "warmup": CPU_WARMUP,
        "repeats": CPU_REPEATS,
        "ssa_diagnostic_draws": 50_000,
        "clock_reference_file": "reference_trajectory_cpu_quick.npz",
        "reference_jit": False,
        "ion_readout_jit": False,
    })

if RUN_CPU_SINGLE_THREAD_DIAGNOSTIC:
    conditions.append({
        "label": "cpu_1thread_diagnostic",
        "device": "cpu",
        "cpu_threads": "1",
        "ion_m_sims": 32_768,
        "clock_num_start_points": 8_192,
        "clock_reference_steps": 600_000,
        "clock_burn_in": 75_000,
        "clock_batch_size": 1_024,
        "warmup": CPU_WARMUP,
        "repeats": CPU_REPEATS,
        "ssa_diagnostic_draws": 50_000,
        "clock_reference_file": "reference_trajectory_cpu_quick.npz",
        "reference_jit": False,
        "ion_readout_jit": False,
    })

print("Configured conditions:")
for c in conditions:
    print(
        f"  {c['label']}: device={c['device']}, threads={c['cpu_threads']}, "
        f"ion={c['ion_m_sims']:,}×20, "
        f"clock={c['clock_batch_size']:,} starts×25×150, "
        f"repeats={c['repeats']}"
    )


In [ ]:
# =============================================================================
# RUN CONDITIONS
# =============================================================================

def output_json_for(c):
    return Path(f"pst_gsst_timing_{c['label']}.json")

def run_condition(c):
    out = output_json_for(c)
    if out.exists() and not FORCE_RERUN:
        print(f"\n=== {c['label']}: existing output found; reusing {out} ===")
        return out

    env = os.environ.copy()
    env.update({
        "PST_BENCH_LABEL": c["label"],
        "PST_BENCH_DEVICE": c["device"],
        "PST_BENCH_GPU_ID": str(GPU_ID),
        "PST_BENCH_CPU_THREADS": str(c["cpu_threads"]),
        "PST_BENCH_WARMUP": str(c["warmup"]),
        "PST_BENCH_REPEATS": str(c["repeats"]),
        "PST_ION_M_SIMS": str(c["ion_m_sims"]),
        "PST_CLOCK_NUM_START_POINTS": str(c["clock_num_start_points"]),
        "PST_CLOCK_REFERENCE_STEPS": str(c["clock_reference_steps"]),
        "PST_CLOCK_BURN_IN": str(c["clock_burn_in"]),
        "PST_CLOCK_BATCH_SIZE": str(c["clock_batch_size"]),
        "PST_SSA_DIAGNOSTIC_DRAWS": str(c["ssa_diagnostic_draws"]),
        "PST_CLOCK_REFERENCE_FILE": str(c["clock_reference_file"]),
        "PST_REFERENCE_JIT": "1" if c["reference_jit"] else "0",
        "PST_ION_READOUT_JIT": "1" if c["ion_readout_jit"] else "0",
        "PST_ION_DATA_FILE": str(ION_DATA_FILE),
        "PST_RUN_ION": "1" if RUN_ION else "0",
        "PST_RUN_CLOCK": "1" if RUN_CLOCK else "0",
        "PST_SHOW_PROGRESS": "1",
    })

    print("\n" + "=" * 96)
    print(f"RUNNING {c['label'].upper()}")
    print("=" * 96, flush=True)

    subprocess.run(
        [sys.executable, str(WORKER_PATH)],
        env=env,
        cwd=".",
        check=True,
    )
    if not out.exists():
        raise RuntimeError(f"Worker completed but did not create {out}")
    return out

output_files = [run_condition(c) for c in conditions]

print("\nCompleted:")
for p in output_files:
    print(" ", p)


Timing results
--------------
model    method       median ms           mean±sd ms    ns/event-eq   M events/s
--------------------------------------------------------------------------------
ion      PST-cat        262.521   260.325±8.521             50.072       19.971
ion      PST-SSA        267.867   266.015±8.399             51.092       19.573
ion      PST-SSA-XLA      259.718   256.429±7.726             49.537       20.187
ion      PST-GM         264.502   264.287±6.028             50.450       19.822
ion      GSST-GM        268.552   267.927±7.037             51.222       19.523
ion      SSA-fwd         32.960    32.863±4.046              6.287      159.070
clock    PST-cat        334.135   334.989±22.934            10.877       91.939
clock    PST-SSA        520.504   516.496±30.709            16.943       59.020
clock    PST-SSA-XLA      391.302   387.181±22.515            12.738       78.507
clock    PST-GM         336.372   334.279±24.063            10.950       91.328
cloc

Timing results
--------------
model    method       median ms           mean±sd ms    ns/event-eq   M events/s
--------------------------------------------------------------------------------
ion      PST-cat       2383.415  2375.227±62.655          3636.803        0.275
ion      PST-SSA       2310.805  2294.755±68.638          3526.008        0.284
ion      PST-SSA-XLA     2294.563  2324.103±269.624         3501.225        0.286
ion      PST-GM        2357.540  2392.146±217.538         3597.321        0.278
ion      GSST-GM       2411.511  2377.897±107.029         3679.674        0.272
ion      SSA-fwd        949.790   969.907±182.130         1449.264        0.690
clock    PST-cat        911.507   908.586±18.287           237.372        4.213
clock    PST-SSA        809.027   807.759±11.602           210.684        4.746
clock    PST-SSA-XLA      741.279   743.796±14.731           193.042        5.180
clock    PST-GM         946.110   940.938±14.171           246.383        4.059
cloc

In [ ]:
# =============================================================================
# COMBINED RESULTS
# =============================================================================

payloads = {
    p.stem.replace("pst_gsst_timing_", ""): json.loads(p.read_text())
    for p in output_files
}

result_rows = []
paired_rows = []

for label, payload in payloads.items():
    env = payload["environment"]
    cfg = payload["configuration"]

    for r in payload["results"]:
        result_rows.append({
            "condition": label,
            "device": cfg.get("benchmark_device", env.get("benchmark_device")),
            "cpu_threads": cfg.get("cpu_threads_requested", env.get("cpu_threads_requested")),
            **r,
        })

    for s in payload["paired_stats"]:
        paired_rows.append({
            "condition": label,
            "device": cfg.get("benchmark_device", env.get("benchmark_device")),
            "cpu_threads": cfg.get("cpu_threads_requested", env.get("cpu_threads_requested")),
            **s,
        })

results_df = pd.DataFrame(result_rows)
paired_df = pd.DataFrame(paired_rows)

display(
    results_df[
        ["condition", "model", "method", "repeats", "median_ms", "mean_ms",
         "std_ms", "ns_per_trajectory_event", "million_trajectory_events_per_s"]
    ].sort_values(["condition", "model", "method"])
)

results_df.to_csv("pst_gsst_gpu_cpu_combined_results.csv", index=False)
paired_df.to_csv("pst_gsst_gpu_cpu_combined_paired_stats.csv", index=False)

print("Saved pst_gsst_gpu_cpu_combined_results.csv")
print("Saved pst_gsst_gpu_cpu_combined_paired_stats.csv")


,condition,model,method,repeats,median_ms,mean_ms,std_ms,ns_per_trajectory_event,million_trajectory_events_per_s
22,cpu_primary,clock,GSST-GM,12,974.045393,974.402410,19.443968,253.657654,3.942321
21,cpu_primary,clock,PST-GM,12,946.109602,940.938268,14.171110,246.382709,4.058726
19,cpu_primary,clock,PST-SSA,12,809.026825,807.759395,11.601905,210.684069,4.746443
20,cpu_primary,clock,PST-SSA-XLA,12,741.279462,743.796108,14.731391,193.041527,5.180233
18,cpu_primary,clock,PST-cat,12,911.506645,908.585926,18.287341,237.371522,4.212805
23,cpu_primary,clock,SSA-fwd,12,323.340583,323.336110,5.925903,84.203277,11.876022
16,cpu_primary,ion,GSST-GM,12,2411.511301,2377.896921,107.028865,3679.674227,0.271763
15,cpu_primary,ion,PST-GM,12,2357.540058,2392.145596,217.537968,3597.320645,0.277985
13,cpu_primary,ion,PST-SSA,12,2310.804593,2294.754588,68.637916,3526.007984,0.283607
14,cpu_primary,ion,PST-SSA-XLA,12,2294.562733,2324.103202,269.623846,3501.224873,0.285614


Saved pst_gsst_gpu_cpu_combined_results.csv
Saved pst_gsst_gpu_cpu_combined_paired_stats.csv


In [ ]:
# =============================================================================
# PRIMARY COMPARISONS
# =============================================================================

PRIMARY_PAIRS = [
    ("PST-GM", "GSST-GM", "matched sampler: backward-rule cost"),
    ("PST-SSA", "GSST-GM", "direct PST implementation vs GS-ST"),
    ("PST-SSA-XLA", "GSST-GM", "XLA direct PST implementation vs GS-ST"),
    ("PST-SSA-XLA", "PST-SSA", "XLA gain within direct PST"),
    ("PST-cat", "GSST-GM", "legacy categorical PST vs GS-ST"),
]

primary_rows = []
for label, payload in payloads.items():
    cfg = payload["configuration"]
    stat_map = {
        (s["model"], s["numerator"], s["denominator"]): s
        for s in payload["paired_stats"]
    }
    models = sorted({r["model"] for r in payload["results"]})

    for model in models:
        for num, den, interpretation in PRIMARY_PAIRS:
            s = stat_map.get((model, num, den))
            if s is None:
                continue
            primary_rows.append({
                "condition": label,
                "device": cfg.get("benchmark_device"),
                "model": model,
                "comparison": f"{num} / {den}",
                "meaning": interpretation,
                "median ratio": s["median_ratio"],
                "95% CI low": s["ratio_ci_low"],
                "95% CI high": s["ratio_ci_high"],
                "paired mean diff ms": s["mean_diff_ms"],
                "n repeats": s["repeats"],
            })

primary_df = pd.DataFrame(primary_rows)
display(primary_df.sort_values(["condition", "model", "comparison"]))
primary_df.to_csv("pst_gsst_gpu_cpu_primary_ratios.csv", index=False)
print("Saved pst_gsst_gpu_cpu_primary_ratios.csv")


,condition,device,model,comparison,meaning,median ratio,95% CI low,95% CI high,paired mean diff ms,n repeats
10,cpu_primary,cpu,clock,PST-GM / GSST-GM,matched sampler: backward-rule cost,0.971320,0.946383,0.991444,-33.464143,12
11,cpu_primary,cpu,clock,PST-SSA / GSST-GM,direct PST implementation vs GS-ST,0.830584,0.815586,0.848047,-166.643015,12
12,cpu_primary,cpu,clock,PST-SSA-XLA / GSST-GM,XLA direct PST implementation vs GS-ST,0.761032,0.751702,0.775094,-230.606302,12
13,cpu_primary,cpu,clock,PST-SSA-XLA / PST-SSA,XLA gain within direct PST,0.916261,0.905180,0.937488,-63.963287,12
14,cpu_primary,cpu,clock,PST-cat / GSST-GM,legacy categorical PST vs GS-ST,0.935795,0.918105,0.952262,-65.816485,12
15,cpu_primary,cpu,ion,PST-GM / GSST-GM,matched sampler: backward-rule cost,0.977619,0.940614,1.029106,14.248674,12
16,cpu_primary,cpu,ion,PST-SSA / GSST-GM,direct PST implementation vs GS-ST,0.958239,0.917326,1.014592,-83.142333,12
17,cpu_primary,cpu,ion,PST-SSA-XLA / GSST-GM,XLA direct PST implementation vs GS-ST,0.951504,0.893287,1.024705,-53.793719,12
18,cpu_primary,cpu,ion,PST-SSA-XLA / PST-SSA,XLA gain within direct PST,0.992971,0.906804,1.057442,29.348614,12
19,cpu_primary,cpu,ion,PST-cat / GSST-GM,legacy categorical PST vs GS-ST,0.988349,0.961184,1.039156,-2.669898,12


Saved pst_gsst_gpu_cpu_primary_ratios.csv


In [ ]:
# =============================================================================
# NORMALIZED COST TABLE
# =============================================================================

# Median training-step cost normalized to GSST-GM within the same device/model.
# SSA-fwd is intentionally excluded here because it is not a training step.
norm_rows = []
for (condition, model), group in results_df.groupby(["condition", "model"]):
    g = group.set_index("method")
    if "GSST-GM" not in g.index:
        continue
    base = float(g.loc["GSST-GM", "median_ms"])
    for method in ["PST-cat", "PST-SSA", "PST-SSA-XLA", "PST-GM", "GSST-GM"]:
        if method in g.index:
            norm_rows.append({
                "condition": condition,
                "model": model,
                "method": method,
                "median ms": float(g.loc[method, "median_ms"]),
                "median / GSST-GM": float(g.loc[method, "median_ms"]) / base,
            })

norm_df = pd.DataFrame(norm_rows)
display(norm_df.sort_values(["condition", "model", "method"]))


,condition,model,method,median ms,median / GSST-GM
4,cpu_primary,clock,GSST-GM,974.045393,1.000000
3,cpu_primary,clock,PST-GM,946.109602,0.971320
1,cpu_primary,clock,PST-SSA,809.026825,0.830584
2,cpu_primary,clock,PST-SSA-XLA,741.279462,0.761032
0,cpu_primary,clock,PST-cat,911.506645,0.935795
9,cpu_primary,ion,GSST-GM,2411.511301,1.000000
8,cpu_primary,ion,PST-GM,2357.540058,0.977619
6,cpu_primary,ion,PST-SSA,2310.804593,0.958239
7,cpu_primary,ion,PST-SSA-XLA,2294.562733,0.951504
5,cpu_primary,ion,PST-cat,2383.415345,0.988349
